# SP-03 Trend-Following Policy — `03_trend_following_policy.ipynb`

Third specialised policy of the **Personalized AI Trading Agent**. A separate Meta-RL model will later choose among five specialised policies; **this model does not select strategies**. It returns one trading action and a confidence for one state.

| Item | Value |
|---|---|
| Model | SP-03 Trend-Following Policy (GRU/LSTM sequence encoder + static branch + dense layer + 4-action head) |
| Action space (shared) | `0 = HOLD`, `1 = BUY`, `2 = SELL`, `3 = EXIT` |
| Main artifact | `trend_following_policy.pt` (+ `trend_following_config.json`, `feature_schema.json`, `preprocessing.pkl`, `evaluation_metrics.json`, `equity_curve.csv`) |
| Execution convention | decision at the **close** of bar *t*, fill at the **open** of bar *t+1* |
| Input schema | same 30 base sequence features and same 6 base static features as SP-01/SP-02, plus 12 trend extras and the SP-02 position extras (unrealized P&L, holding duration) |

> **Research and paper-trading system only.** No profitability is claimed. Trend following is known to lose money in sideways markets and to give back gains at reversals; this notebook reports those failures explicitly (section 9).

### How this differs from the Momentum Policy (SP-01)
SP-01 reacts to *recent directional acceleration* (20/60-bar returns, relative strength, volume confirmation) and its exits are quick. SP-03 is about **regime persistence and position management**:
* it classifies each bar into a **regime** (sideways / uptrend / up-weakening / downtrend / down-weakening) from past-only *structure*: SMA50 vs SMA200, EMA20 vs EMA50, ADX, the R² of a 60-bar log-price regression, Kaufman efficiency ratio and MA slopes;
* it enters when the regime is a *confirmed trend* and price sits high in its 55-bar channel, **holds** while the trend is supported, and exits on a **trend break** or (for impatient profiles) as soon as the trend is merely weakening;
* it uses the position state (unrealized P&L, holding bars) for stops and a minimum holding period.

### Proxy labels are not human demonstrations
By default **no genuine trader actions are supplied**, so the labels are **strategy-derived proxy labels** from the documented rule in section 5 (strategy imitation). They are **not human demonstrations**. Classification scores measure how well the network imitates that rule, nothing more. Set `CONFIG['trader_labels_csv']` if you have real trader actions; then it is true behaviour cloning and the reports say so.

### Notebook map
1 Install · 2 Configuration · 3 Data · 4 Features and regimes · 5 Labels and sequence dataset · 6 Model · 7 Training · 8 Calibration, classification and regime-specific metrics · 9 Backtest, baselines, regime performance and failure analysis · 10 Plots · 11 Export · 12 Inference demo · 13 Meta-RL integration checklist

## 1. Install
Colab already ships PyTorch, pandas, NumPy, scikit-learn and Matplotlib. Only `yfinance` is upgraded. All indicators are implemented manually (no `pandas-ta`, which is fragile on new NumPy/Python versions).

In [ ]:
import subprocess, sys

def _pip_install(*packages):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])
    except Exception as exc:  # noqa: BLE001 - installation problems must not hide the real error later
        print(f"WARNING: pip install {packages} failed: {exc!r}. Continuing with the installed version, if any.")

_pip_install("yfinance")

## 2. Configuration
Everything configurable is in `CONFIG`: symbols, benchmark, dates, chronological split dates, sequence length, model size, regime thresholds, costs, capital and position limits.

* **Splits** are chronological by calendar date; with `strict_window_isolation=True` every validation/test input window also starts inside its split.
* **Long/short** is explicit: `allow_short` controls whether `SELL` (open short) can ever be valid; a long-only variant is always reported when shorts are enabled.
* **Capital and limits:** `initial_capital`, `position_fraction` per entry, `max_position_size` per position and `max_gross_exposure`.
* The regime thresholds in `CONFIG['regime']` are fixed numbers, not fitted on any data.

In [ ]:
import os, json, math, time, random, pickle, warnings, platform, importlib, shutil, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as Fnn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)

warnings.filterwarnings("ignore", category=FutureWarning)

CONFIG = {
    "seed": 42,
    # ---- data ------------------------------------------------------------
    "symbols": ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "JPM", "XOM", "CVX", "JNJ", "PG", "KO", "PEP", "WMT", "V", "HD"],
    "benchmark": "SPY",
    "start_date": "2010-01-01",
    "end_date": "2025-12-31",          # exclusive
    "min_history_rows": 800,           # symbols with fewer clean daily bars are skipped (and reported)
    "local_csv_dir": None,             # optional fallback: <dir>/<SYMBOL>.csv with Date,Open,High,Low,Close,Volume (real data only)
    "trader_labels_csv": None,         # optional genuine labels: columns symbol,date,action,position[,risk_tolerance,patience,aggressiveness]
    # ---- chronological split (calendar dates) ----------------------------
    "train_end": "2020-12-31",
    "val_end": "2022-12-31",
    # ---- sequence / policy -----------------------------------------------
    "seq_len": 30,
    "strict_window_isolation": True,
    "allow_short": True,
    # ---- regimes and labels (all decision-time / past-only) ---------------
    "regime": {"adx_side": 18.0, "er_side": 0.25, "adx_trend": 20.0, "r2_trend": 0.45},
    "min_hold_bars": 2,                  # no regime-based exit before this many bars (stops still apply)
    "hold_sample_max": 60,               # holding duration sampled for position-conditioned training states
    "eval_horizon": 20,                  # bars for the FUTURE-return column used only in evaluation tables
    # ---- backtest --------------------------------------------------------
    "initial_capital": 100000.0,
    "position_fraction": 0.10,         # target notional per new position, fraction of equity
    "max_position_size": 0.20,         # hard cap per position, fraction of equity (applied as min with the target)
    "max_gross_exposure": 1.0,         # sum |notional| <= this * equity (no leverage by default)
    "min_trade_fraction": 0.005,       # smaller entries are skipped
    "fee_bps": 5.0,                    # per side, on traded notional
    "slippage_bps": 5.0,               # per side, adverse price move on every fill
    "short_borrow_bps_annual": 50.0,
    "risk_free_annual": 0.0,
    # ---- model / training -----------------------------------------------
    "model": {"encoder": "gru", "hidden_size": 64, "num_layers": 2, "dropout": 0.25,
              "static_hidden": 16, "head_hidden": 64},
    "train": {"epochs": 40, "batch_size": 512, "lr": 1e-3, "weight_decay": 1e-4, "patience": 7,
              "grad_clip": 1.0, "input_noise_std": 0.02, "max_class_weight": 20.0, "scaler_clip": 5.0},
    "artifact_dir": "/content/sp03_trend_following_artifacts" if Path("/content").exists() else "./sp03_trend_following_artifacts",
}

assert pd.Timestamp(CONFIG["start_date"]) < pd.Timestamp(CONFIG["train_end"]) < pd.Timestamp(CONFIG["val_end"]) < pd.Timestamp(CONFIG["end_date"]), \
    "dates must satisfy start < train_end < val_end < end"

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ART_DIR = Path(CONFIG["artifact_dir"]); ART_DIR.mkdir(parents=True, exist_ok=True)

print(f"python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__} | pandas {pd.__version__}")
print(f"device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else " (CPU fallback)"))
print(f"artifact dir: {ART_DIR.resolve()}")

## 3. Data download
Daily OHLCV from Yahoo Finance via `yfinance` with **adjusted** prices (`auto_adjust=True`, so Open/High/Low/Close are all split/dividend adjusted). Failures are retried, then reported explicitly: a failed symbol is **skipped, never fabricated**. If the benchmark cannot be downloaded or fewer than three symbols succeed, the notebook stops with a clear error.

Cleaning rules (all counted in the report below): duplicate dates removed, non-positive or missing OHLC rows dropped, rows with `High < Low` dropped, negative volume set to missing. Missing volume is kept as missing (never invented) and handled later by explicit train-median imputation.

In [ ]:
import yfinance as yf

OHLCV = ["Open", "High", "Low", "Close", "Volume"]

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy(); df.columns = df.columns.get_level_values(0)
    return df

def download_symbol(sym: str, start: str, end: str, retries: int = 3, local_dir=None):
    """Return (raw_df, source). Raises RuntimeError with the reason if nothing real can be obtained."""
    last_err = "unknown error"
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(sym, start=start, end=end, interval="1d", auto_adjust=True, progress=False, threads=False)
            df = _flatten_columns(df) if df is not None else None
            if df is not None and len(df) > 0:
                return df, "yfinance"
            last_err = "empty result"
        except Exception as exc:  # noqa: BLE001
            last_err = repr(exc)
        time.sleep(1.5 * attempt)
    if local_dir:
        path = Path(local_dir) / f"{sym}.csv"
        if path.exists():
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            return df.loc[start:end], f"local_csv:{path}"
    raise RuntimeError(f"{sym}: download failed after {retries} attempts ({last_err})")

def clean_ohlcv(df: pd.DataFrame, sym: str):
    missing = [c for c in OHLCV if c not in df.columns]
    if missing:
        raise ValueError(f"{sym}: missing columns {missing}")
    stats = {"symbol": sym, "raw_rows": int(len(df))}
    df = df[OHLCV].apply(pd.to_numeric, errors="coerce")
    idx = pd.to_datetime(df.index)
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_localize(None)
    df.index = pd.DatetimeIndex(idx).normalize()
    df.index.name = "Date"
    stats["duplicate_dates_removed"] = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated(keep="last")].sort_index()
    price = ["Open", "High", "Low", "Close"]
    bad = df[price].isna().any(axis=1) | (df[price] <= 0).any(axis=1) | (df["High"] < df["Low"])
    stats["bad_price_rows_dropped"] = int(bad.sum())
    df = df.loc[~bad].copy()
    df.loc[df["Volume"] < 0, "Volume"] = np.nan
    stats["missing_volume_rows"] = int(df["Volume"].isna().sum())
    stats["zero_volume_rows"] = int((df["Volume"] == 0).sum())
    stats["extreme_moves_gt_40pct"] = int((df["Close"].pct_change().abs() > 0.40).sum())
    stats["max_gap_days"] = int(df.index.to_series().diff().dt.days.max()) if len(df) > 1 else 0
    stats["rows"] = int(len(df))
    stats["first_date"] = str(df.index.min().date()) if len(df) else None
    stats["last_date"] = str(df.index.max().date()) if len(df) else None
    return df, stats

download_log, raw_data = [], {}

# benchmark first: it is mandatory
try:
    _b, _src = download_symbol(CONFIG["benchmark"], CONFIG["start_date"], CONFIG["end_date"], local_dir=CONFIG["local_csv_dir"])
    bench_df, _st = clean_ohlcv(_b, CONFIG["benchmark"])
    _st.update(source=_src, status="ok"); download_log.append(_st)
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(f"Benchmark {CONFIG['benchmark']} could not be obtained, cannot continue: {exc}") from exc

for sym in CONFIG["symbols"]:
    try:
        _raw, _src = download_symbol(sym, CONFIG["start_date"], CONFIG["end_date"], local_dir=CONFIG["local_csv_dir"])
        _df, _st = clean_ohlcv(_raw, sym)
        if len(_df) < CONFIG["min_history_rows"]:
            raise ValueError(f"only {len(_df)} clean rows (< min_history_rows={CONFIG['min_history_rows']})")
        _st.update(source=_src, status="ok")
        raw_data[sym] = _df
    except Exception as exc:  # noqa: BLE001
        _st = {"symbol": sym, "status": f"SKIPPED: {exc}"}
        print(f"WARNING: {sym} skipped -> {exc}")
    download_log.append(_st)

if len(raw_data) < 3:
    raise RuntimeError(f"Only {len(raw_data)} symbols downloaded successfully; need at least 3. Check the network / symbols.")

report = pd.DataFrame(download_log).set_index("symbol")
print(f"\nSymbols used ({len(raw_data)}): {list(raw_data)}")
print(f"Symbols skipped: {[s for s in CONFIG['symbols'] if s not in raw_data] or 'none'}")
report

## 4. Feature engineering and regimes
Feature code is written to `sp03_features.py` by the next cell (shipped with the artifacts, no network or training code).

**Sequence input = 42 features per bar, exact order**
* **Shared base (indices 0–29, identical to SP-01 and SP-02):** returns `ret_1, logret_1, ret_5, ret_10, ret_20, ret_60` · `rsi_14, macd_pct, macd_signal_pct, macd_hist_pct` · `close_ema20, close_ema50, ema20_ema50, close_sma200, sma50_sma200` · `adx_14, di_diff` · `atr_pct, rv_20, rv_60` · `vol_chg_1, vol_ratio_20, vol_z_60` · `hl_range, gap_open, close_loc` · `rel_strength_20, rel_strength_60, bench_ret_20, bench_rv_20`.
* **SP-03 trend extras (indices 30–41):** moving-average slopes `sma20_slope_5, sma50_slope_10, sma200_slope_20` · crossovers `ema12_ema26, sma20_sma50` · drawdowns `drawdown_60, drawdown_252` · volume trend `vol_trend_20_60` · persistence `efficiency_ratio_20, trend_r2_60, trend_slope_60` · channel position `donchian_pos_55`.
* Short/medium/long MAs enter through the ratios of price to EMA20/EMA50/SMA200, SMA50/SMA200, SMA20/SMA50 and EMA12/EMA26 (scale-free, so they are stationary).

**Static input = 8 dims (identical to SP-02):** position one-hot `[flat, long, short]` + profile `[risk_tolerance, patience, aggressiveness]` + `unrealized_pnl_scaled` + `holding_bars_scaled` (both 0 when flat). Callers pass raw P&L fraction and bar count.

**Regimes (`classify_regime`, past-only):** `sideways` (ADX < 18 and efficiency ratio < 0.25), `uptrend`, `up_weakening`, `downtrend`, `down_weakening` — exact definitions are in the function's docstring in `sp03_features.py`. The regime of bar *t* uses only data up to the close of *t*; it is used for labels (decision-time) and for grouping evaluation results, never as information from the future.

All features are causal; a perturbation test scrambles all future prices and checks that earlier features do not change.

In [ ]:
%%writefile sp03_features.py
"""SP-03 Trend-Following Policy - shared constants and causal feature engineering.

Schema family rule: the first 30 sequence features and the first 6 static features are identical to SP-01
and SP-02 (same order, same meaning); SP-03 appends its own trend extras. SP-03 static input (8 dims) is
identical to SP-02 (pnl + holding extras). Select policy columns BY NAME from feature_schema.json.

This module never touches the network and never trains anything. Every feature at
row t is computed from data up to and including the close of bar t.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Common action space (shared by all five specialised policies)
# ----------------------------------------------------------------------------
HOLD, BUY, SELL, EXIT = 0, 1, 2, 3
ACTION_NAMES = ["HOLD", "BUY", "SELL", "EXIT"]
N_ACTIONS = 4

# Position state: +1 long, 0 flat, -1 short. One-hot order below is part of the schema.
POSITION_ORDER = (0, 1, -1)                      # one-hot index -> position value
POSITION_NAMES = ["pos_flat", "pos_long", "pos_short"]

# Trader-profile features, each in [0, 1].
PROFILE_FEATURES = ["risk_tolerance", "patience", "aggressiveness"]
DEFAULT_PROFILE = (0.5, 0.5, 0.5)

STATIC_FEATURE_NAMES = POSITION_NAMES + PROFILE_FEATURES   # shared prefix: 3 + 3 = 6 dims

# Sequential market features: exact column order of the model input (30 dims).
BASE_FEATURE_COLUMNS = [
    # returns
    "ret_1", "logret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    # momentum oscillators
    "rsi_14", "macd_pct", "macd_signal_pct", "macd_hist_pct",
    # moving-average relationships
    "close_ema20", "close_ema50", "ema20_ema50", "close_sma200", "sma50_sma200",
    # trend strength
    "adx_14", "di_diff",
    # volatility
    "atr_pct", "rv_20", "rv_60",
    # volume
    "vol_chg_1", "vol_ratio_20", "vol_z_60",
    # candle shape
    "hl_range", "gap_open", "close_loc",
    # relative strength vs benchmark and market context
    "rel_strength_20", "rel_strength_60", "bench_ret_20", "bench_rv_20",
]
# SP-03 additions (appended after the 30 shared features): trend persistence / structure / position management
EXTRA_FEATURE_COLUMNS = [
    "sma20_slope_5", "sma50_slope_10", "sma200_slope_20", "ema12_ema26", "sma20_sma50",
    "drawdown_60", "drawdown_252", "vol_trend_20_60", "efficiency_ratio_20",
    "trend_r2_60", "trend_slope_60", "donchian_pos_55",
]
FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + EXTRA_FEATURE_COLUMNS
N_BASE_FEATURES = len(BASE_FEATURE_COLUMNS)
N_FEATURES = len(FEATURE_COLUMNS)
N_BASE_STATIC = len(STATIC_FEATURE_NAMES)
EXTRA_STATIC_NAMES = ["unrealized_pnl_scaled", "holding_bars_scaled"]
STATIC_FEATURE_NAMES = STATIC_FEATURE_NAMES + EXTRA_STATIC_NAMES
N_STATIC = len(STATIC_FEATURE_NAMES)
PNL_SCALE, PNL_CLIP = 0.10, 3.0       # unrealized_pnl_scaled = clip(pnl / 0.10, -3, 3)
HOLD_SCALE, HOLD_CAP = 20.0, 60       # holding_bars_scaled   = min(bars, 60) / 20
WARMUP_BARS = 260          # drawdown_252 is the longest look-back
OHLCV_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]


# ----------------------------------------------------------------------------
# Action validity and position transitions
# ----------------------------------------------------------------------------
def valid_action_mask(position, allow_short: bool = True) -> np.ndarray:
    """Boolean (B, 4) mask of valid actions given the current position.

    flat  -> HOLD, BUY (open long), SELL (open short, only if allow_short)
    long  -> HOLD, EXIT
    short -> HOLD, EXIT
    """
    pos = np.asarray(position).astype(int).reshape(-1)
    mask = np.zeros((pos.size, N_ACTIONS), dtype=bool)
    mask[:, HOLD] = True
    flat = pos == 0
    mask[flat, BUY] = True
    mask[flat, SELL] = bool(allow_short)
    mask[~flat, EXIT] = True
    return mask


def apply_action(position, action) -> np.ndarray:
    """Return the new position after executing (already validated) actions."""
    pos = np.asarray(position).astype(int).reshape(-1)
    act = np.asarray(action).astype(int).reshape(-1)
    new = pos.copy()
    new[(act == BUY) & (pos == 0)] = 1
    new[(act == SELL) & (pos == 0)] = -1
    new[(act == EXIT) & (pos != 0)] = 0
    return new


def profile_array(profile, batch_size: int) -> np.ndarray:
    """Normalise a profile (dict, sequence, (B,3) array or None) to float32 (B, 3)."""
    if profile is None:
        profile = DEFAULT_PROFILE
    if isinstance(profile, dict):
        profile = [profile.get(k, d) for k, d in zip(PROFILE_FEATURES, DEFAULT_PROFILE)]
    arr = np.asarray(profile, dtype=np.float32)
    if arr.ndim == 1:
        arr = np.tile(arr[None, :], (batch_size, 1))
    if arr.shape != (batch_size, len(PROFILE_FEATURES)):
        raise ValueError(f"profile must have shape (3,) or ({batch_size}, 3); got {arr.shape}")
    if np.any(arr < 0) or np.any(arr > 1) or not np.all(np.isfinite(arr)):
        raise ValueError("profile values must be finite and within [0, 1]")
    return arr


def encode_static(position, profile, unrealized_pnl=None, holding_bars=None) -> np.ndarray:
    """Static input (B, 8) = [flat, long, short] + [risk, patience, aggressiveness] + [pnl_scaled, hold_scaled].

    unrealized_pnl : position-signed fractional P&L since entry (e.g. -0.02 = losing 2%). Ignored when flat.
    holding_bars   : bars the position has been held (entry bar = 1). Ignored when flat.
    """
    pos = np.asarray(position).astype(int).reshape(-1)
    if not np.all(np.isin(pos, POSITION_ORDER)):
        raise ValueError("position must be in {-1, 0, +1}")
    n = pos.size
    onehot = np.stack([(pos == p) for p in POSITION_ORDER], axis=1).astype(np.float32)

    def _vec(v, name):
        if v is None:
            return np.zeros(n, dtype=np.float32)
        a = np.asarray(v, dtype=np.float32).reshape(-1)
        if a.size == 1 and n > 1:
            a = np.repeat(a, n)
        if a.size != n or not np.all(np.isfinite(a)):
            raise ValueError(f"{name} must be a finite scalar or have one entry per state")
        return a

    pnl = np.clip(_vec(unrealized_pnl, "unrealized_pnl") / PNL_SCALE, -PNL_CLIP, PNL_CLIP)
    hold = np.minimum(np.maximum(_vec(holding_bars, "holding_bars"), 0.0), HOLD_CAP) / HOLD_SCALE
    flat = pos == 0
    pnl = np.where(flat, 0.0, pnl); hold = np.where(flat, 0.0, hold)
    return np.concatenate([onehot, profile_array(profile, n), pnl[:, None], hold[:, None]], axis=1).astype(np.float32)


# ----------------------------------------------------------------------------
# Indicators (manual implementations, causal)
# ----------------------------------------------------------------------------
def _pct(s: pd.Series, n: int) -> pd.Series:
    return s / s.shift(n) - 1.0


def _wilder(s: pd.Series, n: int) -> pd.Series:
    return s.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()


def rsi(close: pd.Series, n: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    rs = _wilder(gain, n) / _wilder(loss, n)
    return 100.0 - 100.0 / (1.0 + rs)


def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev = close.shift(1)
    return pd.concat([high - low, (high - prev).abs(), (low - prev).abs()], axis=1).max(axis=1)


def adx_di(high: pd.Series, low: pd.Series, close: pd.Series, n: int = 14):
    """Wilder ADX with +DI / -DI."""
    up = high.diff()
    down = -low.diff()
    plus_dm = pd.Series(np.where((up > down) & (up > 0), up, 0.0), index=high.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=high.index)
    atr = _wilder(true_range(high, low, close), n)
    plus_di = 100.0 * _wilder(plus_dm, n) / atr
    minus_di = 100.0 * _wilder(minus_dm, n) / atr
    dx = 100.0 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0.0, np.nan)
    return _wilder(dx, n), plus_di, minus_di


# ----------------------------------------------------------------------------
# Feature builder
# ----------------------------------------------------------------------------
def build_features(ohlcv: pd.DataFrame, benchmark: pd.DataFrame) -> pd.DataFrame:
    """Compute causal features for one symbol.

    Parameters
    ----------
    ohlcv : DataFrame indexed by date, columns Open/High/Low/Close/Volume (adjusted prices).
    benchmark : DataFrame indexed by date with at least a Close column.

    Returns a DataFrame with the same index: the OHLCV columns followed by FEATURE_COLUMNS.
    Row t only depends on rows <= t (verified by a perturbation test in the notebook).
    """
    if benchmark is None or "Close" not in benchmark:
        raise ValueError("a benchmark DataFrame with a Close column is required")
    missing = [c for c in OHLCV_COLUMNS if c not in ohlcv.columns]
    if missing:
        raise ValueError(f"ohlcv is missing columns: {missing}")

    df = ohlcv[OHLCV_COLUMNS].astype(float).copy()
    o, h, l, c, v = df["Open"], df["High"], df["Low"], df["Close"], df["Volume"]
    f = pd.DataFrame(index=df.index)

    # returns
    f["ret_1"] = _pct(c, 1)
    f["logret_1"] = np.log(c).diff()
    for n in (5, 10, 20, 60):
        f[f"ret_{n}"] = _pct(c, n)

    # momentum oscillators
    f["rsi_14"] = rsi(c, 14)
    ema12 = c.ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = c.ewm(span=26, adjust=False, min_periods=26).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    f["macd_pct"] = macd / c
    f["macd_signal_pct"] = macd_signal / c
    f["macd_hist_pct"] = (macd - macd_signal) / c

    # moving-average relationships
    ema20 = c.ewm(span=20, adjust=False, min_periods=20).mean()
    ema50 = c.ewm(span=50, adjust=False, min_periods=50).mean()
    sma50 = c.rolling(50).mean()
    sma200 = c.rolling(200).mean()
    f["close_ema20"] = c / ema20 - 1.0
    f["close_ema50"] = c / ema50 - 1.0
    f["ema20_ema50"] = ema20 / ema50 - 1.0
    f["close_sma200"] = c / sma200 - 1.0
    f["sma50_sma200"] = sma50 / sma200 - 1.0

    # trend strength
    adx, plus_di, minus_di = adx_di(h, l, c, 14)
    f["adx_14"] = adx
    f["di_diff"] = plus_di - minus_di

    # volatility
    f["atr_pct"] = _wilder(true_range(h, l, c), 14) / c
    f["rv_20"] = f["logret_1"].rolling(20).std() * np.sqrt(252.0)
    f["rv_60"] = f["logret_1"].rolling(60).std() * np.sqrt(252.0)

    # volume (zero volume treated as missing for ratios; never fabricated)
    vz = v.where(v > 0)
    f["vol_chg_1"] = np.log(vz / vz.shift(1)).clip(-3.0, 3.0)
    f["vol_ratio_20"] = vz / vz.rolling(20).mean()
    f["vol_z_60"] = (vz - vz.rolling(60).mean()) / vz.rolling(60).std().replace(0.0, np.nan)

    # candle shape
    f["hl_range"] = (h - l) / c
    f["gap_open"] = o / c.shift(1) - 1.0
    f["close_loc"] = (c - l) / (h - l).replace(0.0, np.nan)

    # relative strength vs benchmark (past-only forward fill, at most 3 bars)
    bc = benchmark["Close"].astype(float).reindex(df.index).ffill(limit=3)
    b_ret20 = _pct(bc, 20)
    b_ret60 = _pct(bc, 60)
    f["rel_strength_20"] = f["ret_20"] - b_ret20
    f["rel_strength_60"] = f["ret_60"] - b_ret60
    f["bench_ret_20"] = b_ret20
    f["bench_rv_20"] = np.log(bc).diff().rolling(20).std() * np.sqrt(252.0)

    # ---- SP-03 trend extras (all causal rolling statistics) -----------------------------------
    sma20 = c.rolling(20).mean()
    f["sma20_slope_5"] = sma20 / sma20.shift(5) - 1.0
    f["sma50_slope_10"] = sma50 / sma50.shift(10) - 1.0
    f["sma200_slope_20"] = sma200 / sma200.shift(20) - 1.0
    f["ema12_ema26"] = ema12 / ema26 - 1.0
    f["sma20_sma50"] = sma20 / sma50 - 1.0
    f["drawdown_60"] = c / c.rolling(60).max() - 1.0
    f["drawdown_252"] = c / c.rolling(252).max() - 1.0
    f["vol_trend_20_60"] = np.log(vz.rolling(20).mean() / vz.rolling(60).mean())
    path = c.diff().abs().rolling(20).sum().replace(0.0, np.nan)
    f["efficiency_ratio_20"] = (c - c.shift(20)).abs() / path           # Kaufman efficiency: 1 = straight line, 0 = noise
    tt = pd.Series(np.arange(len(c), dtype=float), index=c.index)
    logc = np.log(c)
    corr60 = logc.rolling(60).corr(tt)
    f["trend_r2_60"] = corr60 ** 2                                      # linear-trend fit quality of log price
    f["trend_slope_60"] = corr60 * logc.rolling(60).std() / tt.rolling(60).std()   # log-price change per bar
    rng55 = (h.rolling(55).max() - l.rolling(55).min()).replace(0.0, np.nan)
    f["donchian_pos_55"] = (c - l.rolling(55).min()) / rng55            # position inside the 55-bar channel

    f = f[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    return pd.concat([df, f], axis=1)


def scale_features(x, median, mean, std, clip) -> np.ndarray:
    """Single implementation of the preprocessing used in training AND inference.

    Non-finite values -> training-set median, then z-score, then clip to +/- clip.
    """
    x = np.asarray(x, dtype=np.float32)
    x = np.where(np.isfinite(x), x, np.asarray(median, dtype=np.float32))
    x = (x - np.asarray(mean, dtype=np.float32)) / np.asarray(std, dtype=np.float32)
    return np.clip(x, -float(clip), float(clip)).astype(np.float32)


def build_feature_schema(seq_len: int) -> dict:
    """Machine-readable description of the exact model input layout."""
    return {
        "schema_version": "1.0.0",
        "policy_id": "SP-03",
        "sequence_input": {
            "name": "sequence",
            "shape": ["batch", seq_len, N_FEATURES],
            "dtype": "float32",
            "order": "oldest bar first, newest bar last (index -1 is the decision bar)",
            "features": [{"index": i, "name": n} for i, n in enumerate(FEATURE_COLUMNS)],
            "scaling": "NaN -> train median, then (x - mean) / std, then clip to +/- clip (see preprocessing.pkl)",
            "note": "Callers pass RAW (unscaled) features; the policy scales internally.",
        },
        "static_input": {
            "name": "static",
            "shape": ["batch", N_STATIC],
            "extras_note": "indices 0-5 are identical to SP-01/SP-02; indices 6-7 (unrealized P&L, holding duration) are identical to SP-02; default 0 when absent",
            "dtype": "float32",
            "features": [{"index": i, "name": n} for i, n in enumerate(STATIC_FEATURE_NAMES)],
            "position_encoding": {"one_hot_order": list(POSITION_NAMES), "position_values": list(POSITION_ORDER)},
            "profile_ranges": {k: [0.0, 1.0] for k in PROFILE_FEATURES},
            "default_profile": dict(zip(PROFILE_FEATURES, DEFAULT_PROFILE)),
        },
        "output": {
            "action_probs": ["batch", N_ACTIONS],
            "action_space": {str(i): n for i, n in enumerate(ACTION_NAMES)},
            "confidence": "temperature-calibrated probability of the chosen action after masking invalid actions",
        },
        "valid_actions": {
            "flat": ["HOLD", "BUY", "SELL (only if allow_short)"],
            "long": ["HOLD", "EXIT"],
            "short": ["HOLD", "EXIT"],
        },
        "execution_convention": "decision at close of bar t; fill at open of bar t+1",
        "compatibility": {
            "family_rule": "every SP-0x policy consumes the shared base features first, then its own extras; extras DIFFER per policy, so select columns by name",
            "shared_sequence_features": list(BASE_FEATURE_COLUMNS),
            "sp03_extra_sequence_features": list(EXTRA_FEATURE_COLUMNS),
            "shared_static_features": list(STATIC_FEATURE_NAMES[:N_BASE_STATIC]),
            "sp03_extra_static_features": list(EXTRA_STATIC_NAMES),
            "state_dict_keys": ["sequence", "position", "profile", "unrealized_pnl (optional)", "holding_bars (optional)"],
        },
    }


# ----------------------------------------------------------------------------
# Past-only regime classifier (decision-time information only)
# ----------------------------------------------------------------------------
REGIME_NAMES = ["sideways", "uptrend", "up_weakening", "downtrend", "down_weakening"]
DEFAULT_REGIME_PARAMS = {"adx_side": 18.0, "er_side": 0.25, "adx_trend": 20.0, "r2_trend": 0.45}
_REGIME_KEYS = ["adx_14", "efficiency_ratio_20", "sma50_sma200", "ema20_ema50", "close_ema50", "di_diff",
                "trend_r2_60", "sma50_slope_10"]


def classify_regime(feat, params=None) -> np.ndarray:
    """Regime code per row from PAST-ONLY features: -1 unknown, 0 sideways, 1 uptrend, 2 up_weakening,
    3 downtrend, 4 down_weakening. `feat` is a dict / DataFrame with the feature columns.

    sideways      : ADX < adx_side and efficiency ratio < er_side (no directional persistence)
    uptrend       : SMA50 > SMA200, EMA20 > EMA50, close > EMA50, ADX >= adx_trend, +DI > -DI, R^2(60) >= r2_trend, SMA50 rising
    up_weakening  : bullish long-term structure (SMA50 > SMA200) but the uptrend conditions fail (and not sideways)
    downtrend     : mirror of uptrend (SMA50 <= SMA200)
    down_weakening: bearish long-term structure but the downtrend conditions fail (weak / basing / recovering)
    """
    p = {**DEFAULT_REGIME_PARAMS, **(params or {})}
    g = lambda k: np.asarray(feat[k], dtype=float)
    known = np.all([np.isfinite(g(k)) for k in _REGIME_KEYS], axis=0)
    with np.errstate(invalid="ignore"):
        sideways = (g("adx_14") < p["adx_side"]) & (g("efficiency_ratio_20") < p["er_side"])
        up_struct = g("sma50_sma200") > 0
        strong = (g("adx_14") >= p["adx_trend"]) & (g("trend_r2_60") >= p["r2_trend"])
        uptrend = up_struct & (g("ema20_ema50") > 0) & (g("close_ema50") > 0) & strong & (g("di_diff") > 0) & (g("sma50_slope_10") > 0)
        downtrend = (~up_struct) & (g("ema20_ema50") < 0) & (g("close_ema50") < 0) & strong & (g("di_diff") < 0) & (g("sma50_slope_10") < 0)
    reg = np.where(sideways, 0, np.where(uptrend, 1, np.where(up_struct, 2, np.where(downtrend, 3, 4))))
    return np.where(known, reg, -1).astype(np.int64)


In [ ]:
import sp03_features
importlib.reload(sp03_features)
from sp03_features import (ACTION_NAMES, HOLD, BUY, SELL, EXIT, N_ACTIONS, FEATURE_COLUMNS, BASE_FEATURE_COLUMNS,
                           EXTRA_FEATURE_COLUMNS, N_FEATURES, N_BASE_FEATURES, N_STATIC, STATIC_FEATURE_NAMES,
                           PROFILE_FEATURES, DEFAULT_PROFILE, REGIME_NAMES, build_features, build_feature_schema,
                           encode_static, valid_action_mask, apply_action, scale_features, classify_regime)

print(f"{N_FEATURES} sequence features ({N_BASE_FEATURES} shared + {len(EXTRA_FEATURE_COLUMNS)} SP-03 extras), {N_STATIC} static features")
for i, name in enumerate(FEATURE_COLUMNS):
    print(f"  seq[{i:2d}] {name}{'   <- SP-03 extra' if i >= N_BASE_FEATURES else ''}")
for i, name in enumerate(STATIC_FEATURE_NAMES):
    print(f"  static[{i}] {name}{'   <- position extra (same as SP-02)' if i >= 6 else ''}")
print("regimes:", dict(enumerate(REGIME_NAMES)))

In [ ]:
H = CONFIG["eval_horizon"]
EVAL_ONLY_COLUMNS = ["fwd_ret_h"]   # FUTURE prices: evaluation tables only, never a model input and never used for labels
assert set(EVAL_ONLY_COLUMNS).isdisjoint(FEATURE_COLUMNS)

def causality_test(sym: str, frac: float = 0.7) -> None:
    ohlcv = raw_data[sym]; n = int(len(ohlcv) * frac)
    base = build_features(ohlcv, bench_df)[FEATURE_COLUMNS].iloc[:n].to_numpy()
    rng = np.random.default_rng(0)
    pert = ohlcv.copy()
    pert.iloc[n:, :] = pert.iloc[n:, :].to_numpy() * rng.uniform(0.3, 3.0, size=pert.iloc[n:].shape)   # scramble the FUTURE
    bench_p = bench_df.copy()
    future = bench_p.index > ohlcv.index[n - 1]
    bench_p.loc[future, :] = bench_p.loc[future].to_numpy() * 2.0
    alt = build_features(pert, bench_p)[FEATURE_COLUMNS].iloc[:n].to_numpy()
    assert np.allclose(base, alt, equal_nan=True, rtol=1e-9, atol=1e-12), f"{sym}: features depend on future data!"

for _s in list(raw_data)[:2]:
    causality_test(_s)
print("Causality test passed: features at row t are unchanged when all data after t is scrambled.")

frames, feature_report = [], []
for sym, ohlcv in raw_data.items():
    f = build_features(ohlcv, bench_df)
    firsts = [f[c].first_valid_index() for c in FEATURE_COLUMNS]
    if any(x is None for x in firsts):
        print(f"WARNING: {sym} skipped: a feature is entirely missing (history too short for the longest look-back)")
        continue
    f = f.loc[max(firsts):].copy()                                  # drop warm-up rows only
    nan_before = int(f[FEATURE_COLUMNS].isna().sum().sum())
    f[FEATURE_COLUMNS] = f[FEATURE_COLUMNS].ffill(limit=3)          # past-only fill, at most 3 bars
    nan_after = int(f[FEATURE_COLUMNS].isna().sum().sum())
    f["symbol"] = sym
    f["fwd_ret_h"] = f["Open"].shift(-(H + 1)) / f["Open"].shift(-1) - 1.0      # EVAL ONLY (future)
    f = f.reset_index().rename(columns={"Date": "date", "index": "date"})
    feature_report.append({"symbol": sym, "rows_after_warmup": len(f), "first_date": f["date"].min().date(),
                           "nan_cells_before_ffill": nan_before, "nan_cells_after_ffill": nan_after})
    frames.append(f)
if not frames:
    raise RuntimeError("no symbol has enough history for the 252-bar drawdown feature; widen start_date")

panel = pd.concat(frames, ignore_index=True).sort_values(["symbol", "date"]).reset_index(drop=True)
panel["row_in_sym"] = panel.groupby("symbol").cumcount()
panel["regime"] = classify_regime(panel, CONFIG["regime"])            # past-only regime at the close of each bar
print(f"\npanel: {panel.shape[0]:,} rows x {panel.shape[1]} columns, {panel['symbol'].nunique()} symbols, "
      f"{panel['date'].min().date()} -> {panel['date'].max().date()}")
print("Residual NaNs in features are imputed with the TRAIN median at scaling time; rows with unknown regime (-1) are excluded from regime tables.")
_rg = panel[panel["regime"] >= 0]["regime"].map(dict(enumerate(REGIME_NAMES))).value_counts(normalize=True).round(3)
print("regime frequency over all rows:", _rg.to_dict())
pd.DataFrame(feature_report).set_index("symbol")

## 5. Labels and sequence dataset

### 5.1 Decision-time information
Every ingredient of the label (features, regime, position, unrealized P&L, holding bars, profile) is known at the close of bar *t*. **No future outcome is used for labels**, so no label-horizon purge is needed (the future return column `fwd_ret_h` appears only in evaluation tables).

### 5.2 Documented trend-following rule (source of the proxy labels)
Profile-dependent parameters: `adx_entry = 22 − 8·aggressiveness`, `donchian_entry = 0.70 − 0.20·aggressiveness`, `dd_cap = 10% + 10%·risk_tolerance`, `atr_cap = 2% + 4%·risk_tolerance`, `stop = 4% + 8%·risk_tolerance`, `buffer = 1% + 3%·patience`; a profile is **impatient** when `patience < 0.5`.

* **Entry (flat state):** BUY when regime = *uptrend* **and** `ADX ≥ adx_entry` **and** the 55-bar channel position `≥ donchian_entry` **and** `ret_20 > 0` **and** `drawdown_60 ≥ −dd_cap` **and** `ATR% ≤ atr_cap`. SELL (if `allow_short`) is the mirror image in a *downtrend*.
* **Maintain:** while a long is held and none of the exit conditions below hold, the label is **HOLD** (the trend is still supported).
* **Exit (long):** EXIT when unrealized P&L `≤ −stop` (always), or, after `min_hold_bars`, when the trend breaks (`close < EMA50·(1 − buffer)` or regime = *downtrend*), or, **for impatient profiles**, as soon as the regime is no longer a confirmed uptrend (weakening or sideways). Patient profiles ride through weakening and exit only on a break. Shorts mirror this.
* **Position-aware actions:** flat → HOLD/BUY/SELL (SELL only if `allow_short`); long or short → HOLD/EXIT. SELL is therefore *never* an exit; EXIT is the only way to close a position, and impossible actions (EXIT while flat, BUY while long, SELL while long) cannot be labelled or predicted.

### 5.3 Dataset construction and leakage control
* Each bar yields **three** samples (flat / long / short) with a random profile; for held states the holding duration `h ∈ [1, min(hold_sample_max, bars available)]` is sampled and the unrealized P&L is computed from real prices as if filled at the open `h` bars ago, so the position inputs are consistent with the price history.
* Chronological split by date; validation/test windows start inside their own split; shuffling only among training samples.
* The scaler (train median → z-score → clip) is fit on training-period rows only; class imbalance uses inverse-frequency class weights (capped).

In [ ]:
D_TRAIN_END = pd.Timestamp(CONFIG["train_end"]); D_VAL_END = pd.Timestamp(CONFIG["val_end"])
SPLIT_BOUNDS = {"train": (None, D_TRAIN_END),
                "val": (D_TRAIN_END + pd.Timedelta(days=1), D_VAL_END),
                "test": (D_VAL_END + pd.Timedelta(days=1), None)}
L = CONFIG["seq_len"]
RAW = {c: panel[c].to_numpy(dtype=float) for c in FEATURE_COLUMNS}               # unscaled, with NaNs
RAW_MAT = panel[FEATURE_COLUMNS].to_numpy(dtype=np.float32)                      # (N, F)
OPEN_ARR = panel["Open"].to_numpy(dtype=float); CLOSE_ARR = panel["Close"].to_numpy(dtype=float)
FWD = panel["fwd_ret_h"].to_numpy(dtype=float)
REGIME = panel["regime"].to_numpy(dtype=np.int64)
ROW_IN_SYM = panel["row_in_sym"].to_numpy()
_WIN_START = panel.groupby("symbol")["date"].shift(L - 1)                         # first bar of each window

def eligible_rows(split: str) -> np.ndarray:
    """Rows whose decision bar is in the split and whose L-bar window is complete (and, if strict_window_isolation,
    starts inside the split)."""
    start, end = SPLIT_BOUNDS[split]
    ok = _WIN_START.notna().to_numpy().copy()
    if start is not None:
        ok &= (panel["date"] >= start).to_numpy()
        if CONFIG["strict_window_isolation"]:
            ok &= (_WIN_START >= start).to_numpy()
    if end is not None:
        ok &= (panel["date"] <= end).to_numpy()
    return np.where(ok)[0]

def tf_params(profile: np.ndarray) -> dict:
    risk, pat, agg = profile[:, 0], profile[:, 1], profile[:, 2]
    return {"adx_entry": 22.0 - 8.0 * agg, "donch_entry": 0.70 - 0.20 * agg, "dd_cap": 0.10 + 0.10 * risk,
            "atr_cap": 0.02 + 0.04 * risk, "stop": 0.04 + 0.08 * risk, "buffer": 0.01 + 0.03 * pat, "impatient": pat < 0.5}

def trend_rule_labels(feat, pos, profile, hold, pnl, regime, allow_short=True):
    """Documented trend-following rule -> action label (0..3). Decision-time information only."""
    p = tf_params(profile)
    with np.errstate(invalid="ignore"):
        calm = feat["atr_pct"] <= p["atr_cap"]
        long_entry = ((regime == 1) & (feat["adx_14"] >= p["adx_entry"]) & (feat["donchian_pos_55"] >= p["donch_entry"])
                      & (feat["ret_20"] > 0) & (feat["drawdown_60"] >= -p["dd_cap"]) & calm)
        short_entry = ((regime == 3) & (feat["adx_14"] >= p["adx_entry"]) & (feat["donchian_pos_55"] <= 1.0 - p["donch_entry"])
                       & (feat["ret_20"] < 0) & calm)
        settled = hold >= CONFIG["min_hold_bars"]
        known = regime >= 0
        long_break = (feat["close_ema50"] < -p["buffer"]) | (regime == 3)
        short_break = (feat["close_ema50"] > p["buffer"]) | (regime == 1)
        long_weak = p["impatient"] & known & (regime != 1)
        short_weak = p["impatient"] & known & (regime != 3)
        long_exit = (pnl <= -p["stop"]) | (settled & (long_break | long_weak))
        short_exit = (pnl <= -p["stop"]) | (settled & (short_break | short_weak))
    y = np.zeros(len(pos), dtype=np.int64)
    flat = pos == 0
    y[flat & long_entry] = BUY
    if allow_short:
        y[flat & ~long_entry & short_entry] = SELL
    y[(pos == 1) & long_exit] = EXIT
    y[(pos == -1) & short_exit] = EXIT
    return y

def _finalize(rows, pos, prof, hold, pnl, y):
    valid = valid_action_mask(pos, CONFIG["allow_short"])
    assert valid[np.arange(len(y)), y].all(), "label contains an action that is invalid for its position"
    return {"row": np.asarray(rows, dtype=np.int64), "pos": np.asarray(pos, dtype=np.int64),
            "profile": np.asarray(prof, dtype=np.float32), "hold": np.asarray(hold), "pnl": np.asarray(pnl),
            "y": np.asarray(y, dtype=np.int64), "valid": valid, "static": encode_static(pos, prof, pnl, hold)}

def build_rule_samples(split: str, rng: np.random.Generator) -> dict:
    rows = eligible_rows(split)
    r = np.repeat(rows, 3)
    pos = np.tile(np.array([0, 1, -1]), len(rows))
    prof = rng.uniform(0.0, 1.0, size=(len(r), 3)).astype(np.float32)
    cap = np.minimum(CONFIG["hold_sample_max"], ROW_IN_SYM[r] + 1)           # entry bar must exist inside the symbol's history
    hold = np.where(pos == 0, 0, 1 + np.floor(rng.random(len(r)) * cap).astype(int))
    entry_px = OPEN_ARR[r - np.maximum(hold, 1) + 1]                          # hypothetical fill at the open `hold` bars ago
    pnl = np.where(pos == 0, 0.0, pos * (CLOSE_ARR[r] / entry_px - 1.0))
    y = trend_rule_labels({k: v[r] for k, v in RAW.items()}, pos, prof, hold, pnl, REGIME[r], CONFIG["allow_short"])
    return _finalize(r, pos, prof, hold, pnl, y)

def build_trader_samples(split: str, csv_path: str) -> dict:
    """Behaviour cloning from genuine trader actions (optional). Required columns: symbol,date,action,position.
    Optional: risk_tolerance,patience,aggressiveness,unrealized_pnl,holding_bars."""
    tl = pd.read_csv(csv_path)
    need = {"symbol", "date", "action", "position"}
    if not need <= set(tl.columns):
        raise ValueError(f"trader label file needs columns {sorted(need)}; has {list(tl.columns)}")
    tl["date"] = pd.to_datetime(tl["date"]).dt.normalize()
    names = {n: i for i, n in enumerate(ACTION_NAMES)}
    def _act(a):
        s = str(a).strip().upper()
        return names[s] if s in names else int(float(s))
    tl["action"] = tl["action"].map(_act)
    key = panel[["symbol", "date"]].assign(row=np.arange(len(panel)))
    m = tl.merge(key, on=["symbol", "date"], how="inner")
    m = m[m["row"].isin(eligible_rows(split))]
    pos = m["position"].to_numpy(dtype=int)
    if not np.isin(pos, [-1, 0, 1]).all() or not np.isin(m["action"], range(N_ACTIONS)).all():
        raise ValueError("trader labels contain invalid positions or actions")
    ok = valid_action_mask(pos, CONFIG["allow_short"])[np.arange(len(m)), m["action"].to_numpy(dtype=int)]
    if (~ok).any():
        print(f"  [{split}] dropping {(~ok).sum()} trader rows whose action is invalid for the given position")
    m, pos = m[ok], pos[ok]
    if len(m) == 0:
        raise ValueError(f"no usable trader labels in the {split} split")
    prof = (m[PROFILE_FEATURES].to_numpy(dtype=np.float32) if set(PROFILE_FEATURES) <= set(m.columns)
            else np.tile(np.array(DEFAULT_PROFILE, dtype=np.float32), (len(m), 1)))
    pnl = m["unrealized_pnl"].to_numpy(dtype=float) if "unrealized_pnl" in m.columns else np.zeros(len(m))
    hold = m["holding_bars"].to_numpy(dtype=float) if "holding_bars" in m.columns else np.zeros(len(m))
    if ("unrealized_pnl" not in m.columns or "holding_bars" not in m.columns) and (pos != 0).any():
        print(f"  [{split}] NOTE: unrealized_pnl / holding_bars not supplied; held-position rows use 0 for them")
    return _finalize(m["row"].to_numpy(), pos, prof, hold, pnl, m["action"].to_numpy(dtype=int))

rng = np.random.default_rng(CONFIG["seed"])
if CONFIG["trader_labels_csv"]:
    LABEL_SOURCE = "trader_demonstrations"
    SAMPLES = {s: build_trader_samples(s, CONFIG["trader_labels_csv"]) for s in ("train", "val", "test")}
else:
    LABEL_SOURCE = "strategy_derived_proxy (trend rule, decision-time)"
    SAMPLES = {s: build_rule_samples(s, rng) for s in ("train", "val", "test")}

print("=" * 96)
if LABEL_SOURCE.startswith("strategy_derived_proxy"):
    print("LABEL SOURCE: STRATEGY-DERIVED PROXY LABELS from a documented trend-following rule.")
    print("This is STRATEGY IMITATION, not human demonstrations. Classification scores measure imitation of the rule only.")
else:
    print("LABEL SOURCE: genuine trader demonstrations (behaviour cloning).")
print("=" * 96)
dist = pd.DataFrame({s: np.bincount(S["y"], minlength=N_ACTIONS) for s, S in SAMPLES.items()}, index=ACTION_NAMES)
print(dist.assign(**{f"{c}_%": (100 * dist[c] / dist[c].sum()).round(2) for c in dist.columns}))
for s, S in SAMPLES.items():
    r = panel.loc[S["row"], "date"]
    print(f"{s:5s}: {len(S['y']):>9,} samples | decision dates {r.min().date()} -> {r.max().date()}")
assert panel.loc[SAMPLES["train"]["row"], "date"].max() < panel.loc[SAMPLES["val"]["row"], "date"].min()
assert panel.loc[SAMPLES["val"]["row"], "date"].max() < panel.loc[SAMPLES["test"]["row"], "date"].min()
print("chronology check passed: train < validation < test (no overlap of decision dates)")

# regime mix per split (flat-state samples) and what the rule does in each regime
_flat = {s: S["pos"] == 0 for s, S in SAMPLES.items()}
mix = pd.DataFrame({s: pd.Series(REGIME[S["row"][_flat[s]]]).map(lambda v: REGIME_NAMES[v] if v >= 0 else "unknown").value_counts(normalize=True)
                    for s, S in SAMPLES.items()}).fillna(0.0).round(3)
print("\nregime mix by split (share of bars):"); print(mix.to_string())
_tr = SAMPLES["train"]; _ff = _tr["pos"] == 0
entry_by_regime = pd.crosstab(pd.Series(REGIME[_tr["row"][_ff]]).map(lambda v: REGIME_NAMES[v] if v >= 0 else "unknown"),
                              pd.Series(_tr["y"][_ff]).map(dict(enumerate(ACTION_NAMES))))
print("\nTRAIN flat-state labels by regime (proves entries happen only in confirmed trends):"); print(entry_by_regime.to_string())

_cnt = np.bincount(SAMPLES["train"]["y"], minlength=N_ACTIONS).astype(float)
_w = np.where(_cnt > 0, _cnt.sum() / (N_ACTIONS * np.maximum(_cnt, 1)), 0.0)
_w = np.minimum(_w, CONFIG["train"]["max_class_weight"])
CLASS_WEIGHTS = (_w * (_cnt.sum() / (_w * _cnt).sum())).astype(np.float32)
print("\nclass weights:", {n: round(float(w), 3) for n, w in zip(ACTION_NAMES, CLASS_WEIGHTS)})
if (_cnt == 0).any():
    print("NOTE: classes with no training examples:", [ACTION_NAMES[i] for i in np.where(_cnt == 0)[0]])

# ---- preprocessing fit on TRAIN-PERIOD rows only ----------------------------------------------------------
_fit = (panel["date"] <= D_TRAIN_END).to_numpy()
_med = np.nanmedian(RAW_MAT[_fit], axis=0)
_imp = np.where(np.isfinite(RAW_MAT[_fit]), RAW_MAT[_fit], _med)
_std = _imp.std(axis=0); _std = np.where(_std < 1e-8, 1.0, _std)
PREPROCESSING = {"feature_columns": list(FEATURE_COLUMNS), "median": _med.astype(np.float32),
                 "mean": _imp.mean(axis=0).astype(np.float32), "std": _std.astype(np.float32),
                 "clip": float(CONFIG["train"]["scaler_clip"]),
                 "fit_rows_end_date": str(D_TRAIN_END.date()), "n_fit_rows": int(_fit.sum())}
X_SCALED = scale_features(RAW_MAT, PREPROCESSING["median"], PREPROCESSING["mean"], PREPROCESSING["std"], PREPROCESSING["clip"])
print(f"scaler fit on {PREPROCESSING['n_fit_rows']:,} rows with dates <= {PREPROCESSING['fit_rows_end_date']}; "
      f"non-finite feature cells imputed overall: {int((~np.isfinite(RAW_MAT)).sum()):,}")

## 6. Model definition
The network lives in `sp03_model.py` (written by the next cell and shipped with the artifacts); it contains **no data-download or training code**.

* `TrendFollowingPolicyNet`: GRU or LSTM encoder over the last `seq_len` bars → last hidden state, concatenated with a dense branch for the 8-dim static input (position, profile, unrealized P&L, holding bars) → dense layer → 4 action logits. `seq_len`, `hidden_size`, `num_layers`, `encoder` live in `CONFIG`.
* `TrendFollowingPolicy`: `predict(state)`, `save(path)`, `load(path, device)`. It scales raw features with the saved preprocessing, masks impossible actions by position and returns **temperature-calibrated** probabilities.
* **Confidence** = calibrated probability of the chosen action among the *valid* actions. It is calibrated against the training labels, so it means "agreement with the rule that produced the labels", not "probability this trade is profitable".

**`predict(state)` input** (same keys as SP-01/SP-02): `{"sequence": (L, 42) or (B, L, 42) RAW features, "position": -1|0|+1, "profile": [risk, patience, aggressiveness], "unrealized_pnl": float (optional), "holding_bars": int (optional)}`.

In [ ]:
%%writefile sp03_model.py
"""SP-03 Trend-Following Policy - network and inference wrapper.

Contains NO data download and NO training code. Load a saved policy with
``TrendFollowingPolicy.load(dir_or_pt_file, device)`` and call ``predict(state)``.
"""
from __future__ import annotations

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sp03_features import (
    ACTION_NAMES, DEFAULT_PROFILE, FEATURE_COLUMNS, N_ACTIONS, N_FEATURES, N_STATIC,
    PROFILE_FEATURES, WARMUP_BARS, build_feature_schema, build_features, encode_static,
    scale_features, valid_action_mask,
)

ARTIFACT_NAME = "trend_following_policy.pt"
CONFIG_NAME = "trend_following_config.json"
MASK_VALUE = -1e9


class TrendFollowingPolicyNet(nn.Module):
    """GRU/LSTM encoder over market features + static (position, profile) branch + action head."""

    def __init__(self, n_features: int = N_FEATURES, n_static: int = N_STATIC, hidden_size: int = 64,
                 num_layers: int = 2, dropout: float = 0.25, static_hidden: int = 16,
                 head_hidden: int = 64, encoder: str = "gru", n_actions: int = N_ACTIONS):
        super().__init__()
        if encoder not in ("gru", "lstm"):
            raise ValueError("encoder must be 'gru' or 'lstm'")
        rnn_cls = nn.GRU if encoder == "gru" else nn.LSTM
        self.rnn = rnn_cls(n_features, hidden_size, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.static = nn.Sequential(nn.Linear(n_static, static_hidden), nn.GELU())
        self.head = nn.Sequential(
            nn.Linear(hidden_size + static_hidden, head_hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(head_hidden, n_actions),
        )

    def forward(self, seq: torch.Tensor, static: torch.Tensor) -> torch.Tensor:
        out, _ = self.rnn(seq)
        z = torch.cat([out[:, -1, :], self.static(static)], dim=1)
        return self.head(z)


def mask_logits(logits: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    return logits.masked_fill(~valid, MASK_VALUE)


def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


class TrendFollowingPolicy:
    """Inference wrapper: scaling + network + action masking + calibrated confidence."""

    def __init__(self, net: TrendFollowingPolicyNet, config: dict, preprocessing: dict, device="cpu"):
        self.device = torch.device(device)
        self.net = net.to(self.device).eval()
        self.config = _jsonable(config)
        self.pre = {
            "feature_columns": list(preprocessing["feature_columns"]),
            "median": np.asarray(preprocessing["median"], dtype=np.float32),
            "mean": np.asarray(preprocessing["mean"], dtype=np.float32),
            "std": np.asarray(preprocessing["std"], dtype=np.float32),
            "clip": float(preprocessing["clip"]),
        }
        if self.pre["feature_columns"] != list(FEATURE_COLUMNS):
            raise ValueError("feature order in preprocessing does not match sp03_features.FEATURE_COLUMNS")
        self.seq_len = int(self.config["seq_len"])
        self.allow_short = bool(self.config.get("allow_short", True))
        self.temperature = float(self.config.get("temperature", 1.0))

    # ------------------------------------------------------------------ scaling
    def scale(self, seq: np.ndarray) -> np.ndarray:
        """Raw (..., F) -> scaled float32. NaN -> train median, standardise, clip."""
        return scale_features(seq, self.pre["median"], self.pre["mean"], self.pre["std"], self.pre["clip"])

    # ------------------------------------------------------------------ inference
    @torch.no_grad()
    def predict_scaled(self, seq_scaled: torch.Tensor, static: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        """Calibrated probabilities (B, 4) from already-scaled tensors on self.device."""
        logits = mask_logits(self.net(seq_scaled, static), valid)
        return torch.softmax(logits / self.temperature, dim=1)

    @torch.no_grad()
    def predict(self, state: dict) -> dict:
        """Predict an action for one state or a batch of states.

        state keys
        ----------
        sequence : array (L, 30) or (B, L, 30) of RAW features in FEATURE_COLUMNS order,
                   oldest bar first. L >= seq_len; the last seq_len bars are used.
        position : int in {-1, 0, 1} (or length-B array). Default 0 (flat).
        profile  : dict or length-3 / (B, 3) array [risk_tolerance, patience, aggressiveness]
                   in [0, 1]. Default (0.5, 0.5, 0.5).
        unrealized_pnl : (optional) position-signed fractional P&L since entry, scalar or per-state. Default 0.
        holding_bars   : (optional) bars held, entry bar = 1, scalar or per-state. Default 0.
                   Both are ignored (zeroed) when position == 0.

        Returns a dict with action_probs, action, action_name, confidence, valid_actions, entropy.
        Single-state input yields scalars / 1-D arrays; batch input yields arrays.
        """
        seq = np.asarray(state["sequence"], dtype=np.float32)
        single = seq.ndim == 2
        if single:
            seq = seq[None]
        if seq.ndim != 3 or seq.shape[2] != N_FEATURES:
            raise ValueError(f"sequence must have shape (L, {N_FEATURES}) or (B, L, {N_FEATURES}); got {seq.shape}")
        if seq.shape[1] < self.seq_len:
            raise ValueError(f"need at least seq_len={self.seq_len} bars; got {seq.shape[1]}")
        seq = seq[:, -self.seq_len:, :]
        batch = seq.shape[0]

        position = np.asarray(state.get("position", 0)).astype(int).reshape(-1)
        if position.size == 1 and batch > 1:
            position = np.repeat(position, batch)
        if position.size != batch:
            raise ValueError("position must be scalar or have one entry per sequence")
        static = encode_static(position, state.get("profile", DEFAULT_PROFILE),
                               state.get("unrealized_pnl"), state.get("holding_bars"))
        valid = valid_action_mask(position, self.allow_short)

        seq_t = torch.from_numpy(self.scale(seq)).to(self.device)
        static_t = torch.from_numpy(static).to(self.device)
        valid_t = torch.from_numpy(valid).to(self.device)
        probs = self.predict_scaled(seq_t, static_t, valid_t).cpu().numpy()

        action = probs.argmax(axis=1)
        confidence = probs[np.arange(batch), action]
        entropy = -(probs * np.log(np.clip(probs, 1e-12, 1.0))).sum(axis=1)
        names = [ACTION_NAMES[a] for a in action]
        if single:
            return {"action_probs": probs[0], "action": int(action[0]), "action_name": names[0],
                    "confidence": float(confidence[0]), "valid_actions": valid[0], "entropy": float(entropy[0])}
        return {"action_probs": probs, "action": action, "action_name": names,
                "confidence": confidence, "valid_actions": valid, "entropy": entropy}

    def state_from_ohlcv(self, ohlcv: pd.DataFrame, benchmark: pd.DataFrame, position: int = 0, profile=None,
                         unrealized_pnl: float = 0.0, holding_bars: int = 0) -> dict:
        """Build a predict() state from OHLCV frames the caller already holds (no downloading)."""
        feats = build_features(ohlcv, benchmark)[FEATURE_COLUMNS]
        if len(feats) < WARMUP_BARS + self.seq_len:
            raise ValueError(f"need at least {WARMUP_BARS + self.seq_len} bars of history; got {len(feats)}")
        window = feats.iloc[-self.seq_len:]
        return {"sequence": window.to_numpy(dtype=np.float32), "position": int(position),
                "profile": profile if profile is not None else DEFAULT_PROFILE,
                "unrealized_pnl": unrealized_pnl, "holding_bars": holding_bars}

    # ------------------------------------------------------------------ persistence
    def save(self, path) -> Path:
        """Write trend_following_policy.pt, trend_following_config.json, feature_schema.json, preprocessing.pkl into `path`.

        trend_following_policy.pt is self-contained and authoritative; the other three are standard
        side-car copies for tooling and the Meta-RL selector.
        """
        out = Path(path)
        out.mkdir(parents=True, exist_ok=True)
        schema = build_feature_schema(self.seq_len)
        pre_plain = {"feature_columns": self.pre["feature_columns"], "median": self.pre["median"].tolist(),
                     "mean": self.pre["mean"].tolist(), "std": self.pre["std"].tolist(), "clip": self.pre["clip"]}
        config = dict(self.config)
        config["temperature"] = self.temperature
        bundle = {
            "format_version": 1,
            "state_dict": {k: v.detach().cpu() for k, v in self.net.state_dict().items()},
            "config": config,
            "preprocessing": pre_plain,
            "feature_schema": schema,
        }
        torch.save(bundle, out / ARTIFACT_NAME)
        (out / CONFIG_NAME).write_text(json.dumps(_jsonable(config), indent=2))
        (out / "feature_schema.json").write_text(json.dumps(_jsonable(schema), indent=2))
        with open(out / "preprocessing.pkl", "wb") as fh:
            pickle.dump({**pre_plain, "median": self.pre["median"], "mean": self.pre["mean"], "std": self.pre["std"]}, fh)
        return out / ARTIFACT_NAME

    @classmethod
    def load(cls, path, device=None) -> "TrendFollowingPolicy":
        """Load from a directory containing trend_following_policy.pt, or from the .pt file itself."""
        p = Path(path)
        pt = p / ARTIFACT_NAME if p.is_dir() else p
        if not pt.exists():
            raise FileNotFoundError(f"policy file not found: {pt}")
        device = torch.device(device) if device is not None else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        bundle = torch.load(pt, map_location=device, weights_only=True)
        config = bundle["config"]
        net = TrendFollowingPolicyNet(**config["model"])
        net.load_state_dict(bundle["state_dict"])
        return cls(net, config, bundle["preprocessing"], device)


In [ ]:
import sp03_model
importlib.reload(sp03_model)
from sp03_model import TrendFollowingPolicyNet, TrendFollowingPolicy, mask_logits

set_seed(CONFIG["seed"])
MODEL_CFG = {"n_features": N_FEATURES, "n_static": N_STATIC, "n_actions": N_ACTIONS, **CONFIG["model"]}
net = TrendFollowingPolicyNet(**MODEL_CFG).to(DEVICE)
print(net)
print(f"trainable parameters: {sum(p.numel() for p in net.parameters() if p.requires_grad):,}")

## 7. Training
Supervised classification of the (proxy or genuine) action labels with a class-weighted cross-entropy on **position-masked logits**. Early stopping uses the validation loss; the best weights are restored. Windows are gathered on the fly from the scaled feature matrix (no giant window tensor is stored).

In [ ]:
X_ALL = torch.from_numpy(X_SCALED).to(DEVICE)
OFFSETS = torch.arange(-(L - 1), 1, device=DEVICE)
OFFSETS_NP = np.arange(-(L - 1), 1)

def to_torch(S):
    return {"row": torch.from_numpy(S["row"]).long().to(DEVICE), "static": torch.from_numpy(S["static"]).to(DEVICE),
            "y": torch.from_numpy(S["y"]).long().to(DEVICE), "valid": torch.from_numpy(S["valid"]).to(DEVICE)}

TS = {k: to_torch(v) for k, v in SAMPLES.items()}

def get_batch(T, idx):
    win = T["row"][idx].unsqueeze(1) + OFFSETS            # (B, L) global row ids, all inside one symbol block
    return X_ALL[win], T["static"][idx], T["valid"][idx], T["y"][idx]

@torch.no_grad()
def forward_all(model, T, batch_size=4096):
    """Masked logits for every sample of a split (CPU tensor)."""
    model.eval(); out = []
    for i in range(0, len(T["y"]), batch_size):
        idx = torch.arange(i, min(i + batch_size, len(T["y"])), device=DEVICE)
        seq, st, valid, _ = get_batch(T, idx)
        out.append(mask_logits(model(seq, st), valid).cpu())
    return torch.cat(out)

tc = CONFIG["train"]
class_w = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)
optimizer = torch.optim.AdamW(net.parameters(), lr=tc["lr"], weight_decay=tc["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
y_val_cpu = TS["val"]["y"].cpu()
gen = torch.Generator().manual_seed(CONFIG["seed"])
n_train = len(TS["train"]["y"])

history, best_val, best_state, bad_epochs = [], float("inf"), None, 0
t0 = time.time()
for epoch in range(1, tc["epochs"] + 1):
    net.train()
    perm = torch.randperm(n_train, generator=gen).to(DEVICE)       # shuffling happens inside the TRAIN split only
    total, seen = 0.0, 0
    for i in range(0, n_train, tc["batch_size"]):
        idx = perm[i:i + tc["batch_size"]]
        seq, st, valid, y = get_batch(TS["train"], idx)
        if tc["input_noise_std"] > 0:
            seq = seq + tc["input_noise_std"] * torch.randn_like(seq)
        loss = Fnn.cross_entropy(mask_logits(net(seq, st), valid), y, weight=class_w)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), tc["grad_clip"])
        optimizer.step()
        total += loss.item() * len(idx); seen += len(idx)
    train_loss = total / seen
    val_logits = forward_all(net, TS["val"])
    val_loss = Fnn.cross_entropy(val_logits, y_val_cpu, weight=class_w.cpu()).item()
    val_f1 = f1_score(y_val_cpu.numpy(), val_logits.argmax(1).numpy(), average="macro", labels=list(range(N_ACTIONS)), zero_division=0)
    scheduler.step(val_loss)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_macro_f1": val_f1,
                    "lr": optimizer.param_groups[0]["lr"]})
    improved = val_loss < best_val - 1e-5
    if improved:
        best_val, bad_epochs = val_loss, 0
        best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    else:
        bad_epochs += 1
    print(f"epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f} | val macro-F1 {val_f1:.3f} | lr {optimizer.param_groups[0]['lr']:.1e}"
          f"{'  *' if improved else ''}  [{time.time() - t0:.0f}s]")
    if bad_epochs >= tc["patience"]:
        print(f"early stopping at epoch {epoch} (no validation improvement for {tc['patience']} epochs)")
        break

net.load_state_dict(best_state); net.eval()
history_df = pd.DataFrame(history)
BEST_EPOCH = int(history_df.loc[history_df["val_loss"].idxmin(), "epoch"])
print(f"\nbest epoch: {BEST_EPOCH} | best validation loss: {best_val:.4f}")

## 8. Calibration, classification metrics and regime-specific evaluation
* **Temperature scaling** is fit on the validation logits only and then frozen.
* Metrics on the chronological **validation** and **test** splits: accuracy, balanced accuracy, macro/weighted F1, per-class precision/recall/F1, confusion matrix, NLL and Expected Calibration Error.
* These measure **imitation of the proxy labels** (unless you supplied trader actions). The majority-class baseline is printed for reference.
* **Regime-specific evaluation:** the same predictions are broken down by the (past-only) regime of the decision bar and by position state, so you can see where the network follows the rule and where it does not.

In [ ]:
def fit_temperature(logits: torch.Tensor, y: torch.Tensor) -> float:
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad()
        loss = Fnn.cross_entropy(logits / torch.exp(log_t), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.exp(log_t).clamp(0.05, 10.0).item())

def ece_score(conf, correct, bins=10):
    edges = np.linspace(0, 1, bins + 1); ece, rows = 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
            rows.append({"bin_lo": lo, "bin_hi": hi, "n": int(m.sum()), "confidence": float(conf[m].mean()), "accuracy": float(correct[m].mean())})
    return float(ece), rows

def classification_summary(logits, y, temperature):
    y = np.asarray(y); probs = torch.softmax(logits / temperature, dim=1).numpy()
    pred = probs.argmax(1); conf = probs.max(1); labels = list(range(N_ACTIONS))
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    ece, bins = ece_score(conf, (pred == y).astype(float))
    return {"n_samples": int(len(y)), "accuracy": float(accuracy_score(y, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=labels, zero_division=0)),
            "weighted_f1": float(f1_score(y, pred, average="weighted", labels=labels, zero_division=0)),
            "majority_class_baseline_accuracy": float(np.bincount(y, minlength=N_ACTIONS).max() / len(y)),
            "nll": float(-np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1.0)).mean()),
            "ece": ece, "reliability_bins": bins, "temperature": float(temperature),
            "per_class": {ACTION_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])} for i in labels},
            "confusion_matrix": confusion_matrix(y, pred, labels=labels).tolist(), "pred": pred}

LOGITS = {k: forward_all(net, TS[k]) for k in ("val", "test")}
Y = {k: TS[k]["y"].cpu().numpy() for k in ("val", "test")}
TEMPERATURE = fit_temperature(LOGITS["val"], TS["val"]["y"].cpu())
print(f"fitted temperature (validation only): {TEMPERATURE:.3f}")

CLS = {k: classification_summary(LOGITS[k], Y[k], TEMPERATURE) for k in ("val", "test")}
CLS_UNCAL_TEST = classification_summary(LOGITS["test"], Y["test"], 1.0)
for k in ("val", "test"):
    c = CLS[k]
    print(f"\n[{k}] acc {c['accuracy']:.3f} | balanced acc {c['balanced_accuracy']:.3f} | macro-F1 {c['macro_f1']:.3f} | "
          f"weighted-F1 {c['weighted_f1']:.3f} | majority baseline {c['majority_class_baseline_accuracy']:.3f}")
    print(f"      NLL {c['nll']:.4f} | ECE {c['ece']:.4f}" + (f"  (uncalibrated test: NLL {CLS_UNCAL_TEST['nll']:.4f}, ECE {CLS_UNCAL_TEST['ece']:.4f})" if k == "test" else ""))
    print(pd.DataFrame(c["per_class"]).T.round(3).to_string())
    print("confusion matrix (rows = true, cols = predicted):")
    print(pd.DataFrame(c["confusion_matrix"], index=ACTION_NAMES, columns=ACTION_NAMES).to_string())


policy_config = {
    "model_name": "SP-03 Trend-Following Policy", "policy_id": "SP-03", "version": "1.0.0", "artifact": "trend_following_policy.pt",
    "seq_len": L, "allow_short": CONFIG["allow_short"], "temperature": TEMPERATURE,
    "model": MODEL_CFG, "action_space": {str(i): n for i, n in enumerate(ACTION_NAMES)},
    "label_source": LABEL_SOURCE,
    "label_note": ("Labels are strategy-derived PROXY labels from a documented trend-following rule: strategy imitation, NOT human demonstrations."
                   if LABEL_SOURCE.startswith("strategy_derived_proxy") else "Labels are genuine trader demonstrations (behaviour cloning)."),
    "label_definition": {"information": "decision-time (past-only) features, regime, position, unrealized P&L, holding bars, profile",
                         "future_outcomes_used": False, "regime_params": CONFIG["regime"], "min_hold_bars": CONFIG["min_hold_bars"]},
    "confidence_definition": "temperature-calibrated probability of the chosen action after masking invalid actions; calibrated against the training labels, not against trade profitability",
    "execution_convention": "decision at close of bar t, fill at open of bar t+1",
    "regime_names": dict(enumerate(REGIME_NAMES)),
    "shared_schema": {"n_shared_sequence_features": N_BASE_FEATURES, "n_extra_sequence_features": len(EXTRA_FEATURE_COLUMNS),
                      "n_shared_static_features": 6, "n_extra_static_features": 2,
                      "note": "base 30 sequence columns identical to SP-01/SP-02; extras differ per policy (select columns by name)"},
    "data": {"symbols": list(raw_data), "benchmark": CONFIG["benchmark"], "start_date": CONFIG["start_date"], "end_date": CONFIG["end_date"],
             "train_end": CONFIG["train_end"], "val_end": CONFIG["val_end"], "source": "yfinance daily adjusted OHLCV"},
    "training": {**CONFIG["train"], "seed": CONFIG["seed"], "best_epoch": BEST_EPOCH, "class_weights": CLASS_WEIGHTS.tolist(),
                 "n_train_samples": int(n_train)},
    "intended_use": "research and paper trading only; not a recommendation to trade real money",
}
policy = TrendFollowingPolicy(net, policy_config, PREPROCESSING, DEVICE)
print("\nTrendFollowingPolicy ready (in-memory).")


In [ ]:
# ---- regime-specific and position-specific classification quality (validation and test) -------------------------
LABELS4 = list(range(N_ACTIONS))
REG_CLS = {}
for split in ("val", "test"):
    S = SAMPLES[split]; y = Y[split]; pred = CLS[split]["pred"]; reg = REGIME[S["row"]]; rows_ = []
    for code_, name in list(enumerate(REGIME_NAMES)) + [(-1, "unknown")]:
        m = reg == code_
        if m.sum() == 0:
            continue
        present = np.unique(np.concatenate([y[m], pred[m]]))
        rows_.append({"regime": name, "n": int(m.sum()), "accuracy": float((y[m] == pred[m]).mean()),
                      "macro_f1_present_classes": float(f1_score(y[m], pred[m], average="macro", labels=present, zero_division=0)),
                      **{f"true_{a}": int((y[m] == i).sum()) for i, a in enumerate(ACTION_NAMES)},
                      **{f"pred_{a}": int((pred[m] == i).sum()) for i, a in enumerate(ACTION_NAMES)}})
    REG_CLS[split] = pd.DataFrame(rows_).set_index("regime")
    pos_rows = []
    for pv, pn in ((0, "flat"), (1, "long"), (-1, "short")):
        m = S["pos"] == pv
        if m.sum():
            pos_rows.append({"position": pn, "n": int(m.sum()), "accuracy": float((y[m] == pred[m]).mean()),
                             "macro_f1_present_classes": float(f1_score(y[m], pred[m], average="macro", labels=np.unique(np.concatenate([y[m], pred[m]])), zero_division=0))})
    REG_CLS[split + "_by_position"] = pd.DataFrame(pos_rows).set_index("position")
    print(f"\n[{split}] classification by regime of the decision bar (all position states):")
    print(REG_CLS[split].round(3).to_string())
    print(f"[{split}] classification by position state:")
    print(REG_CLS[split + "_by_position"].round(3).to_string())
print("\nRead with care: regime rows with few BUY/SELL labels cannot support conclusions about entries.")

## 9. Backtest, baselines, regime performance and failure analysis
A small portfolio simulator with explicit cash, shares and constraints (the same engine as SP-02):

* **Timing:** the policy sees bar *t* (close known) and its orders are filled at the **open of bar t+1**.
* **Costs:** every fill is moved against us by `slippage_bps`; `fee_bps` is charged on traded notional; shorts pay `short_borrow_bps_annual`.
* **Capital and limits:** an entry targets `min(position_fraction, max_position_size) × equity`, limited by available cash (longs) and `max_gross_exposure`; unfundable entries are skipped and counted. Exits run first; competing entries are ranked by confidence. Open positions are marked to market at the end, not liquidated.
* **Long/short:** the main run follows `CONFIG['allow_short']`; when shorts are on, a **long-only** variant is also reported.
* **Comparators:** SPY buy-and-hold, equal-weight universe buy-and-hold (daily rebalanced, no costs), the **trend rule applied directly** and a simple documented baseline: the **SMA50/SMA200 golden cross** (long-only: BUY when flat and SMA50 > SMA200 and close > SMA200; EXIT when SMA50 < SMA200), run through the same engine with the same costs and limits.
* **Regime performance:** daily returns are grouped by the **SPY regime known at the previous close** (past-only), so "trending" and "sideways" periods are identified without hindsight.
* **Failures:** drawdown episodes, calendar-year results, worst 21-day windows, whipsaw trades and worst trades are printed for the main run and the benchmark, not just the best period.

Nothing here assumes trend following is profitable. The test split is untouched by training, calibration and early stopping, but it is one regime and a short window.

In [ ]:
PROFILES = {"balanced": (0.5, 0.5, 0.5), "conservative": (0.2, 0.8, 0.2), "aggressive": (0.8, 0.3, 0.8)}
FEE, SLIP = CONFIG["fee_bps"] / 1e4, CONFIG["slippage_bps"] / 1e4
BORROW_DAILY = CONFIG["short_borrow_bps_annual"] / 1e4 / 252.0
CAP0 = float(CONFIG["initial_capital"])
OFFSETS_NP = np.arange(-(L - 1), 1)

def prepare_backtest(split: str) -> dict:
    rows = eligible_rows(split)                                         # no outcome needed: backtests never use labels
    sub = panel.loc[rows, ["symbol", "date"]].assign(row=rows)
    piv = sub.pivot(index="date", columns="symbol", values="row")
    syms = list(piv.columns)
    n_all = len(piv); piv = piv.dropna(how="any")
    if len(piv) < n_all:
        print(f"  [{split}] {n_all - len(piv)} dates dropped because not every symbol had a complete window")
    rows_mat = piv[syms].to_numpy().astype(np.int64)
    bclose = bench_df["Close"].reindex(piv.index).ffill(limit=3).to_numpy(dtype=float)
    if np.isnan(bclose).any():
        raise RuntimeError("benchmark close has gaps inside the backtest window")
    return {"split": split, "dates": piv.index, "symbols": syms, "rows": rows_mat,
            "open": OPEN_ARR[rows_mat], "close": CLOSE_ARR[rows_mat], "bench_close": bclose}

def run_portfolio_backtest(bt: dict, decide_fn, allow_short: bool) -> dict:
    T, S = bt["open"].shape
    cash = CAP0; shares = np.zeros(S); entry_px = np.zeros(S); entry_k = np.zeros(S, dtype=int); hold = np.zeros(S, dtype=int)
    prev_eq = CAP0
    eq_hist, ret_hist, expo_hist, turn_hist, trades = [], [], [], [], []
    n_entries = n_skipped = 0
    for k in range(T - 1):
        pos = np.sign(shares).astype(int)
        pnl = np.where(pos != 0, pos * (bt["close"][k] / np.where(entry_px > 0, entry_px, 1.0) - 1.0), 0.0)
        probs = np.asarray(decide_fn(k, bt["rows"][k], pos, pnl, hold), dtype=float)       # decision at the close of bar k
        valid = valid_action_mask(pos, allow_short)
        action = np.where(valid, probs, -1.0).argmax(axis=1)
        conf = probs[np.arange(S), action]
        o = bt["open"][k + 1]; traded = 0.0; entered = np.zeros(S, dtype=bool)
        for s in np.where(action == EXIT)[0]:                                               # exits first: free capital
            sh = shares[s]
            px = o[s] * (1 - SLIP) if sh > 0 else o[s] * (1 + SLIP)
            cash += sh * px - FEE * abs(sh) * px
            traded += abs(sh) * px
            trades.append({"symbol": bt["symbols"][s], "side": int(np.sign(sh)), "entry_date": bt["dates"][entry_k[s]],
                           "exit_date": bt["dates"][k + 1], "decision_date": bt["dates"][entry_k[s] - 1], "bars": int(hold[s]), "ret": float(np.sign(sh) * (px / entry_px[s] - 1.0))})
            shares[s] = 0.0; entry_px[s] = 0.0
        equity_open = cash + float((shares * o).sum())
        for s in [i for i in np.argsort(-conf) if action[i] in (BUY, SELL)]:                # entries, most confident first
            target = min(CONFIG["position_fraction"], CONFIG["max_position_size"]) * equity_open
            room = CONFIG["max_gross_exposure"] * equity_open - float(np.abs(shares * o).sum())
            if action[s] == BUY:
                px = o[s] * (1 + SLIP); notional = min(target, room, cash / (1 + FEE))
            else:
                px = o[s] * (1 - SLIP); notional = min(target, room)
            if notional < CONFIG["min_trade_fraction"] * equity_open:
                n_skipped += 1; continue
            if action[s] == BUY:
                shares[s] = notional / px; cash -= notional * (1 + FEE)
            else:
                shares[s] = -notional / px; cash += notional * (1 - FEE)
            entry_px[s] = px; entry_k[s] = k + 1; entered[s] = True; traded += notional; n_entries += 1
        c1 = bt["close"][k + 1]
        short_mask = shares < 0
        cash -= float(-(shares[short_mask] * c1[short_mask]).sum()) * BORROW_DAILY
        eq = max(cash + float((shares * c1).sum()), 1e-9)
        hold = np.where(entered, 1, np.where(shares != 0, hold + 1, 0))
        ret_hist.append(eq / prev_eq - 1.0); eq_hist.append(eq / CAP0)
        expo_hist.append(float(np.abs(shares * c1).sum()) / eq); turn_hist.append(traded / prev_eq)
        prev_eq = eq
    tr = pd.DataFrame(trades)
    return {"dates": bt["dates"][1:], "ret": np.array(ret_hist), "equity": np.array(eq_hist), "exposure": np.array(expo_hist),
            "turnover": np.array(turn_hist), "n_entries": n_entries, "n_closed_trades": len(tr), "n_skipped_entries": n_skipped,
            "open_positions_at_end": int((shares != 0).sum()), "trades": tr,
            "win_rate": float((tr["ret"] > 0).mean()) if len(tr) else float("nan"),
            "avg_trade_return": float(tr["ret"].mean()) if len(tr) else float("nan")}

def perf_metrics(r, turnover=None, exposure=None) -> dict:
    r = np.asarray(r, dtype=float); n = len(r); eq = np.cumprod(1.0 + r)
    years = n / 252.0; sd = r.std(ddof=1) if n > 1 else np.nan
    ex = r - CONFIG["risk_free_annual"] / 252.0
    dd_sd = np.sqrt((np.minimum(ex, 0.0) ** 2).mean()) if n else np.nan
    peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    mdd = float((eq / peak - 1.0).min())
    cagr = float(eq[-1] ** (1 / years) - 1) if years > 0 and eq[-1] > 0 else float("nan")
    out = {"n_days": int(n), "total_return": float(eq[-1] - 1.0), "cagr": cagr,
           "ann_vol": float(sd * np.sqrt(252)) if sd == sd else float("nan"),
           "sharpe": float(ex.mean() / sd * np.sqrt(252)) if sd and sd > 0 else float("nan"),
           "sortino": float(ex.mean() / dd_sd * np.sqrt(252)) if dd_sd and dd_sd > 0 else float("nan"),
           "max_drawdown": mdd, "calmar": float(cagr / abs(mdd)) if mdd < 0 and cagr == cagr else float("nan")}
    out["annualized_turnover"] = float(np.mean(turnover) * 252.0) if turnover is not None else 0.0
    out["avg_gross_exposure"] = float(np.mean(exposure)) if exposure is not None else 1.0
    return out


def model_decide(profile):
    prof = np.asarray(profile, dtype=np.float32)
    def fn(k, rows, pos, pnl, hold):
        seq = RAW_MAT[rows[:, None] + OFFSETS_NP]                              # raw past-only windows (S, L, F)
        return policy.predict({"sequence": seq, "position": pos, "profile": prof,
                               "unrealized_pnl": pnl, "holding_bars": hold})["action_probs"]
    return fn

def rule_decide(profile, allow_short):
    """The documented trend rule applied directly (decision-time information only)."""
    prof = np.asarray(profile, dtype=np.float32)
    def fn(k, rows, pos, pnl, hold):
        y = trend_rule_labels({c: v[rows] for c, v in RAW.items()}, pos, np.tile(prof, (len(rows), 1)), hold, pnl,
                              REGIME[rows], allow_short)
        return np.eye(N_ACTIONS)[y]
    return fn

def golden_cross_decide(k, rows, pos, pnl, hold):
    """Baseline: long-only SMA50/SMA200 golden cross. BUY if flat and SMA50 > SMA200 and close > SMA200; EXIT if SMA50 < SMA200."""
    spread = RAW["sma50_sma200"][rows]; c200 = RAW["close_sma200"][rows]
    y = np.zeros(len(rows), dtype=int)
    with np.errstate(invalid="ignore"):
        y[(pos == 0) & (spread > 0) & (c200 > 0)] = BUY
        y[(pos == 1) & (spread < 0)] = EXIT
    return np.eye(N_ACTIONS)[y]

LS = "long/short" if CONFIG["allow_short"] else "long-only"
MAIN = f"SP-03 model (balanced, {LS})"
LONG_ONLY = "SP-03 model (balanced, long-only)"
RULE = f"Trend rule (balanced, {LS})"
GC = "SMA50/200 golden cross (long-only)"
BT, BT_RESULTS, BT_TABLE = {}, {}, {}
for split in ("val", "test"):
    bt = prepare_backtest(split); BT[split] = bt
    runs = {}
    for pname, prof in PROFILES.items():
        runs[f"SP-03 model ({pname}, {LS})"] = run_portfolio_backtest(bt, model_decide(prof), CONFIG["allow_short"])
    if CONFIG["allow_short"]:
        runs[LONG_ONLY] = run_portfolio_backtest(bt, model_decide(PROFILES["balanced"]), False)
    runs[RULE] = run_portfolio_backtest(bt, rule_decide(PROFILES["balanced"], CONFIG["allow_short"]), CONFIG["allow_short"])
    runs[GC] = run_portfolio_backtest(bt, golden_cross_decide, False)
    dates = runs[MAIN]["dates"]
    b_ret = bt["bench_close"][1:] / bt["bench_close"][:-1] - 1.0
    ew_ret = (bt["close"][1:] / bt["close"][:-1] - 1.0).mean(axis=1)
    metrics = {n: perf_metrics(r["ret"], r["turnover"], r["exposure"]) for n, r in runs.items()}
    metrics[f"{CONFIG['benchmark']} buy & hold"] = perf_metrics(b_ret)
    metrics["Equal-weight universe buy & hold"] = perf_metrics(ew_ret)
    BT_RESULTS[split] = {"runs": runs, "bench_ret": b_ret, "ew_ret": ew_ret, "dates": dates}
    extra = pd.DataFrame({n: {"trade_entries": r["n_entries"], "closed_trades": r["n_closed_trades"], "skipped_entries": r["n_skipped_entries"],
                              "win_rate": r["win_rate"], "avg_trade_return": r["avg_trade_return"], "open_at_end": r["open_positions_at_end"]}
                          for n, r in runs.items()}).T
    BT_TABLE[split] = pd.DataFrame(metrics).T.join(extra)
    print(f"\n===== {split.upper()} backtest: {dates[0].date()} -> {dates[-1].date()} ({len(dates)} days, {len(bt['symbols'])} symbols, capital {CAP0:,.0f}) =====")
    print(f"fee {CONFIG['fee_bps']} bps + slippage {CONFIG['slippage_bps']} bps per side | size {CONFIG['position_fraction']:.0%} (cap {CONFIG['max_position_size']:.0%}) per position | "
          f"max gross {CONFIG['max_gross_exposure']:.0%} | shorts allowed: {CONFIG['allow_short']}")
    print(BT_TABLE[split][["total_return", "cagr", "sharpe", "max_drawdown", "annualized_turnover", "avg_gross_exposure",
                           "trade_entries", "win_rate", "skipped_entries"]].round(3).to_string())


In [ ]:
# ---- Performance in trending vs sideways periods (regime of SPY known at the PREVIOUS close) ----------------------
bench_feat = build_features(bench_df, bench_df)          # benchmark features (relative-strength columns are not used by the regime rule)
bench_regime = pd.Series(classify_regime(bench_feat, CONFIG["regime"]), index=bench_feat.index)
GROUPS = {"GROUP trending (up or down)": [1, 3], "GROUP sideways": [0], "GROUP weakening / transition": [2, 4]}
REGIME_PERF = {}
for split in ("val", "test"):
    R = BT_RESULTS[split]; dts = BT[split]["dates"]
    rg = bench_regime.reindex(dts[:-1]).fillna(-1).to_numpy().astype(int)      # regime at close t applies to the return of day t+1
    series = {n: r["ret"] for n, r in R["runs"].items() if n in (MAIN, LONG_ONLY, RULE, GC)}
    series[f"{CONFIG['benchmark']} buy & hold"] = R["bench_ret"]; series["Equal-weight universe buy & hold"] = R["ew_ret"]
    rows_ = []
    for name, r in series.items():
        for gname, codes in list(GROUPS.items()) + [(REGIME_NAMES[c], [c]) for c in range(len(REGIME_NAMES))]:
            m = np.isin(rg, codes)
            if m.sum() < 5:
                continue
            rr = r[m]; sd = rr.std(ddof=1)
            rows_.append({"series": name, "regime": gname, "days": int(m.sum()), "ann_return": float(rr.mean() * 252),
                          "ann_vol": float(sd * np.sqrt(252)), "sharpe": float(rr.mean() / sd * np.sqrt(252)) if sd > 0 else np.nan,
                          "compounded_return": float(np.prod(1 + rr) - 1), "hit_rate": float((rr > 0).mean())})
    REGIME_PERF[split] = pd.DataFrame(rows_)
    share = pd.Series(rg).map(lambda v: REGIME_NAMES[v] if v >= 0 else "unknown").value_counts(normalize=True).round(3)
    print(f"\n===== {split.upper()}: performance by SPY regime (known at the previous close) =====")
    print("share of days by regime:", share.to_dict())
    piv = REGIME_PERF[split][REGIME_PERF[split]["regime"].isin(GROUPS)].pivot(index="series", columns="regime", values="ann_return")
    print("annualised return by regime group:"); print(piv.round(3).to_string())
    pv2 = REGIME_PERF[split][REGIME_PERF[split]["regime"].isin(GROUPS)].pivot(index="series", columns="regime", values="sharpe")
    print("Sharpe by regime group:"); print(pv2.round(2).to_string())

# ---- Trades grouped by the regime in which they were entered (main run, test) -------------------------------------
_idx = pd.Series(np.arange(len(panel)), index=pd.MultiIndex.from_arrays([panel["symbol"], panel["date"]]))
def trades_by_entry_regime(tr: pd.DataFrame) -> pd.DataFrame:
    if len(tr) == 0:
        return pd.DataFrame()
    ix = _idx.reindex(pd.MultiIndex.from_arrays([tr["symbol"], pd.to_datetime(tr["decision_date"])])).to_numpy()
    reg = np.where(np.isnan(ix), -1, REGIME[np.nan_to_num(ix).astype(int)])
    t = tr.assign(entry_regime=[REGIME_NAMES[v] if v >= 0 else "unknown" for v in reg], side_name=np.where(tr["side"] > 0, "long", "short"))
    return t.groupby(["entry_regime", "side_name"]).agg(trades=("ret", "size"), win_rate=("ret", lambda s: (s > 0).mean()),
                                                       mean_return=("ret", "mean"), median_bars=("bars", "median"))
TRADE_REGIME = {s: trades_by_entry_regime(BT_RESULTS[s]["runs"][MAIN]["trades"]) for s in ("val", "test")}
print("\nTEST: closed trades of the main run by entry regime:")
print(TRADE_REGIME["test"].round(3).to_string() if len(TRADE_REGIME["test"]) else "no closed trades in the test window")

In [ ]:
# ---- Failure analysis: drawdowns, bad years, worst windows, whipsaws, worst trades (TEST) --------------------------
def drawdown_episodes(eq, dates, top=3):
    eq = np.asarray(eq, dtype=float); peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    dd = eq / peak - 1.0; eps, in_dd = [], False
    for t in range(len(eq)):
        if dd[t] < 0 and not in_dd:
            in_dd, start, trough = True, max(t - 1, 0), t
        if in_dd:
            if dd[t] < dd[trough]:
                trough = t
            if dd[t] >= 0:
                eps.append((start, trough, t)); in_dd = False
    if in_dd:
        eps.append((start, trough, None))
    rows_ = []
    for s, tr_, e in sorted(eps, key=lambda x: dd[x[1]])[:top]:
        rows_.append({"peak_date": dates[s].date(), "trough_date": dates[tr_].date(), "depth": float(dd[tr_]),
                      "days_peak_to_trough": int(tr_ - s), "recovered_on": dates[e].date() if e is not None else "NOT RECOVERED",
                      "days_underwater": int((e if e is not None else len(eq) - 1) - s)})
    return pd.DataFrame(rows_)

R = BT_RESULTS["test"]; dts = R["dates"]
named = {MAIN: R["runs"][MAIN]["ret"], GC: R["runs"][GC]["ret"], f"{CONFIG['benchmark']} buy & hold": R["bench_ret"]}
if LONG_ONLY in R["runs"]:
    named[LONG_ONLY] = R["runs"][LONG_ONLY]["ret"]
FAIL = {"drawdown_episodes": {}, "calendar_years": None, "worst_21d": {}, "whipsaw": {}, "worst_trades": None}
for name, r in named.items():
    eq = np.cumprod(1 + r)
    FAIL["drawdown_episodes"][name] = drawdown_episodes(eq, dts)
    print(f"\n--- worst drawdown episodes: {name} ---")
    print(FAIL["drawdown_episodes"][name].to_string(index=False))
yearly = pd.DataFrame({n: pd.Series(r, index=dts).groupby(dts.year).apply(lambda x: np.prod(1 + x) - 1) for n, r in named.items()})
FAIL["calendar_years"] = yearly
print("\ncalendar-year compounded returns (partial years are partial):"); print(yearly.round(3).to_string())
for name, r in named.items():
    w = pd.Series(r).rolling(21).apply(lambda x: np.prod(1 + x) - 1, raw=True).dropna()
    FAIL["worst_21d"][name] = float(w.min()) if len(w) else float("nan")
print("worst 21-day compounded return:", {k: round(v, 4) for k, v in FAIL["worst_21d"].items()})
tr = R["runs"][MAIN]["trades"]
if len(tr):
    short_ = tr[tr["bars"] <= 5]
    FAIL["whipsaw"] = {"n_trades": int(len(tr)), "share_closed_within_5_bars": float(len(short_) / len(tr)),
                       "mean_return_of_those": float(short_["ret"].mean()) if len(short_) else float("nan"),
                       "share_losing": float((tr["ret"] < 0).mean()), "mean_loss": float(tr.loc[tr["ret"] < 0, "ret"].mean()) if (tr["ret"] < 0).any() else float("nan"),
                       "mean_win": float(tr.loc[tr["ret"] > 0, "ret"].mean()) if (tr["ret"] > 0).any() else float("nan")}
    FAIL["worst_trades"] = tr.nsmallest(5, "ret")[["symbol", "side", "decision_date", "exit_date", "bars", "ret"]]
    print("\nwhipsaw / trade-quality stats (main run, test):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in FAIL["whipsaw"].items()})
    print("worst 5 closed trades:"); print(FAIL["worst_trades"].to_string(index=False))
else:
    print("\nno closed trades in the test window for the main run (nothing to analyse; see skipped/open counts above)")
print("\nDo not average these away: a trend policy that only wins in strong trends will show its losses in the sideways rows and drawdown tables above.")

In [ ]:
# ---- Do trends actually persist? Executable 20-bar forward return by regime and by model decision (TEST, flat state) ----
test_rows = eligible_rows("test")
preds = []
for i in range(0, len(test_rows), 2048):
    r = test_rows[i:i + 2048]
    preds.append(policy.predict({"sequence": RAW_MAT[r[:, None] + OFFSETS_NP], "position": 0, "profile": PROFILES["balanced"]})["action"])
preds = np.concatenate(preds)
fwd = FWD[test_rows]; reg_t = REGIME[test_rows]; okf = np.isfinite(fwd)
rows_ = []
for code_, name in enumerate(REGIME_NAMES):
    m = (reg_t == code_) & okf
    if m.sum() == 0:
        continue
    b, s = m & (preds == BUY), m & (preds == SELL)
    rows_.append({"regime": name, "bars": int(m.sum()), "mean_fwd_return_all": float(fwd[m].mean()),
                  "model_BUY_n": int(b.sum()), "mean_fwd_after_BUY": float(fwd[b].mean()) if b.any() else np.nan,
                  "model_SELL_n": int(s.sum()), "mean_fwd_after_SELL_dir_adj": float(-fwd[s].mean()) if s.any() else np.nan})
exec_table = pd.DataFrame(rows_).set_index("regime")
print(f"Executable {H}-bar forward return (next open -> open {H} bars later) by regime and model decision, flat state, TEST:")
print(exec_table.round(4).to_string())
print("\nA trend policy needs mean_fwd_after_BUY (and dir-adjusted SELL) to beat the unconditional column. Small n means no statistical conclusion.")

## 10. Plots

In [ ]:
FIG_DIR = ART_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_df["epoch"], history_df["train_loss"], label="train loss")
ax[0].plot(history_df["epoch"], history_df["val_loss"], label="validation loss")
ax[0].axvline(BEST_EPOCH, color="grey", ls="--", lw=1, label=f"best epoch ({BEST_EPOCH})")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("weighted cross-entropy"); ax[0].set_title("Learning curves"); ax[0].legend()
ax[1].plot(history_df["epoch"], history_df["val_macro_f1"], color="tab:green")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("macro F1"); ax[1].set_title("Validation macro-F1")
plt.tight_layout(); plt.savefig(FIG_DIR / "learning_curves.png", dpi=130); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
cm = np.array(CLS["test"]["confusion_matrix"]); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
for a, mat, title, fmt in ((ax[0], cm, "Test confusion matrix (counts)", "d"), (ax[1], cmn, "Test confusion matrix (row-normalised)", ".2f")):
    a.imshow(mat, cmap="Blues"); a.set_title(title)
    a.set_xticks(range(N_ACTIONS)); a.set_yticks(range(N_ACTIONS)); a.set_xticklabels(ACTION_NAMES); a.set_yticklabels(ACTION_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true (label)")
    for i in range(N_ACTIONS):
        for j in range(N_ACTIONS):
            a.text(j, i, format(mat[i, j], fmt), ha="center", va="center", color="white" if mat[i, j] > mat.max() / 2 else "black")
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix_test.png", dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(4.8, 4.5))
bins = pd.DataFrame(CLS["test"]["reliability_bins"])
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
ax.plot(bins["confidence"], bins["accuracy"], "o-", label=f"test (ECE {CLS['test']['ece']:.3f})")
ax.set_xlabel("mean confidence"); ax.set_ylabel("accuracy vs labels"); ax.set_title("Reliability diagram"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "reliability_test.png", dpi=130); plt.show()

R = BT_RESULTS["test"]; main = R["runs"][MAIN]; d = R["dates"]
REG_COLORS = {0: "#bbbbbb", 1: "#7fc97f", 2: "#d9ef8b", 3: "#f4a582", 4: "#fddbc7"}
rg_plot = bench_regime.reindex(BT["test"]["dates"][:-1]).fillna(-1).to_numpy().astype(int)

def shade(a):
    start = 0
    for i in range(1, len(rg_plot) + 1):
        if i == len(rg_plot) or rg_plot[i] != rg_plot[start]:
            if rg_plot[start] in REG_COLORS:
                a.axvspan(d[start], d[min(i, len(d) - 1)], color=REG_COLORS[rg_plot[start]], alpha=0.25, lw=0)
            start = i

fig, ax = plt.subplots(3, 1, figsize=(11, 11), sharex=True, gridspec_kw={"height_ratios": [3, 1.6, 1.4]})
shade(ax[0])
ax[0].plot(d, main["equity"], label=MAIN, lw=2)
if LONG_ONLY in R["runs"]:
    ax[0].plot(d, R["runs"][LONG_ONLY]["equity"], label=LONG_ONLY, lw=1.4)
ax[0].plot(d, R["runs"][GC]["equity"], label=GC, lw=1.2)
ax[0].plot(d, R["runs"][RULE]["equity"], label=RULE, lw=1.0, alpha=0.8)
ax[0].plot(d, np.cumprod(1 + R["bench_ret"]), label=f"{CONFIG['benchmark']} buy & hold", lw=1.2)
ax[0].plot(d, np.cumprod(1 + R["ew_ret"]), label="Equal-weight universe buy & hold", lw=1.2)
ax[0].set_ylabel("growth of 1.0"); ax[0].set_title("TEST split (out-of-sample): equity after costs. Shading = SPY regime (green up-trend, grey sideways, orange down-trend)")
ax[0].legend(fontsize=8)
for name, eq in ((MAIN, main["equity"]), (GC, R["runs"][GC]["equity"]), (CONFIG["benchmark"], np.cumprod(1 + R["bench_ret"]))):
    ax[1].plot(d, eq / np.maximum.accumulate(eq) - 1, label=name)
ax[1].set_ylabel("drawdown"); ax[1].legend(fontsize=8)
ax[2].plot(d, main["exposure"], color="tab:purple"); ax[2].set_ylabel("gross exposure"); ax[2].set_ylim(0, max(1.05, main["exposure"].max() * 1.05))
plt.tight_layout(); plt.savefig(FIG_DIR / "backtest_test.png", dpi=130); plt.show()

rp = REGIME_PERF["test"]; rp = rp[rp["regime"].isin(GROUPS)]
fig, ax = plt.subplots(figsize=(9, 4.2))
pv = rp.pivot(index="regime", columns="series", values="ann_return").reindex(list(GROUPS))
pv.plot(kind="bar", ax=ax); ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("annualised return within regime"); ax.set_title("TEST: return by SPY regime group"); plt.xticks(rotation=0)
ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG_DIR / "regime_performance_test.png", dpi=130); plt.show()

sym_plot = list(raw_data)[0]
pp = panel[(panel["symbol"] == sym_plot) & (panel["date"] > D_VAL_END)]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(pp["date"], pp["Close"], lw=1, label="close")
ax.plot(pp["date"], pp["Close"] / (1 + pp["close_ema50"]), lw=1, label="EMA50")
ax.plot(pp["date"], pp["Close"] / (1 + pp["close_sma200"]), lw=1, label="SMA200")
tr = main["trades"]; tr = tr[tr["symbol"] == sym_plot] if len(tr) else tr
for _, t in tr.iterrows():
    ax.axvspan(t["entry_date"], t["exit_date"], color="green" if t["side"] > 0 else "red", alpha=0.12, lw=0)
ax.set_title(f"{sym_plot}: closed trades of the main run on the test split (green = long, red = short)"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "trades_example.png", dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
(dist / dist.sum() * 100).plot(kind="bar", ax=ax); ax.set_ylabel("% of samples"); ax.set_title("Label distribution"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(FIG_DIR / "label_distribution.png", dpi=130); plt.show()

## 11. Save standardized artifacts
| File | Content |
|---|---|
| `trend_following_policy.pt` | weights + config + preprocessing + feature schema (self-contained, loads with `weights_only=True`) |
| `trend_following_config.json` | model / inference configuration (side-car copy) |
| `feature_schema.json` | exact input ordering and dimensions, incl. the cross-policy compatibility block |
| `preprocessing.pkl` | train-fit median / mean / std / clip (plain dict, no custom classes) |
| `evaluation_metrics.json` | classification, calibration, regime, backtest, baseline and failure metrics, with the proxy-label warning |
| `equity_curve.csv` | chronological equity curves (validation and test, each starting at 1.0) with the SPY regime |
| `sp03_features.py`, `sp03_model.py` | the exact code needed to use the policy |

`trend_following_policy.pt` is authoritative; the other copies are for inspection and tooling.

In [ ]:
def to_py(o):
    if isinstance(o, pd.DataFrame): return to_py(o.reset_index().to_dict(orient="records"))
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.ndarray): return to_py(o.tolist())
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (pd.Timestamp,)) or hasattr(o, "isoformat"): return str(o)
    return o

pt_path = policy.save(ART_DIR)       # trend_following_policy.pt, trend_following_config.json, feature_schema.json, preprocessing.pkl

def strip_pred(c): return {k: v for k, v in c.items() if k != "pred"}
evaluation = {
    "model": "SP-03 Trend-Following Policy", "label_source": LABEL_SOURCE, "label_warning": policy_config["label_note"],
    "scores_interpretation": ("Classification scores measure imitation of a rule-derived label set (strategy imitation), "
                              "not human behaviour and not profitability." if LABEL_SOURCE.startswith("strategy_derived_proxy")
                              else "Classification scores measure behaviour cloning of the supplied trader actions."),
    "disclaimer": "Research and paper-trading system only. Backtest numbers are one historical path and are not evidence of future profitability.",
    "label_definition": policy_config["label_definition"], "regime_definition": CONFIG["regime"],
    "data": policy_config["data"], "n_samples": {k: int(len(v["y"])) for k, v in SAMPLES.items()},
    "label_distribution": dist.to_dict(), "class_weights": dict(zip(ACTION_NAMES, CLASS_WEIGHTS.tolist())),
    "classification": {"validation": strip_pred(CLS["val"]), "test": strip_pred(CLS["test"]), "test_uncalibrated": strip_pred(CLS_UNCAL_TEST)},
    "classification_by_regime": {k: v for k, v in REG_CLS.items()},
    "backtest_assumptions": {"execution": "decision at close t, fill at open t+1", "fee_bps": CONFIG["fee_bps"], "slippage_bps": CONFIG["slippage_bps"],
                             "short_borrow_bps_annual": CONFIG["short_borrow_bps_annual"], "risk_free_annual": CONFIG["risk_free_annual"],
                             "allow_short": CONFIG["allow_short"], "initial_capital": CONFIG["initial_capital"],
                             "position_fraction": CONFIG["position_fraction"], "max_position_size": CONFIG["max_position_size"],
                             "max_gross_exposure": CONFIG["max_gross_exposure"], "min_trade_fraction": CONFIG["min_trade_fraction"],
                             "baseline": "SMA50/SMA200 golden cross, long-only, same engine and costs"},
    "backtest": {s: BT_TABLE[s].to_dict(orient="index") for s in ("val", "test")},
    "backtest_by_spy_regime": {s: REGIME_PERF[s] for s in ("val", "test")},
    "trades_by_entry_regime": {s: TRADE_REGIME[s] for s in ("val", "test")},
    "failure_analysis_test": {"drawdown_episodes": FAIL["drawdown_episodes"], "calendar_years": FAIL["calendar_years"],
                              "worst_21d_return": FAIL["worst_21d"], "whipsaw": FAIL["whipsaw"], "worst_trades": FAIL["worst_trades"]},
    "forward_return_by_regime_test": exec_table,
    "training_history": history_df.to_dict(orient="records"), "best_epoch": BEST_EPOCH,
}
(ART_DIR / "evaluation_metrics.json").write_text(json.dumps(to_py(evaluation), indent=2))

curves = []
for split in ("val", "test"):
    R_ = BT_RESULTS[split]; m_ = R_["runs"][MAIN]
    rg_ = bench_regime.reindex(BT[split]["dates"][:-1]).to_numpy()
    df_ = pd.DataFrame({"date": R_["dates"], "split": split, "strategy_return": m_["ret"], "strategy_equity": m_["equity"],
                        "benchmark_return": R_["bench_ret"], "benchmark_equity": np.cumprod(1 + R_["bench_ret"]),
                        "equal_weight_bh_equity": np.cumprod(1 + R_["ew_ret"]), "golden_cross_equity": R_["runs"][GC]["equity"],
                        "trend_rule_equity": R_["runs"][RULE]["equity"], "gross_exposure": m_["exposure"], "turnover": m_["turnover"],
                        "spy_regime_prev_close": [REGIME_NAMES[int(v)] if v == v and v >= 0 else "unknown" for v in rg_]})
    if LONG_ONLY in R_["runs"]:
        df_["long_only_equity"] = R_["runs"][LONG_ONLY]["equity"]
    curves.append(df_)
pd.concat(curves, ignore_index=True).to_csv(ART_DIR / "equity_curve.csv", index=False)

for fname in ("sp03_features.py", "sp03_model.py"):
    shutil.copy(fname, ART_DIR / fname)

reloaded = TrendFollowingPolicy.load(ART_DIR, DEVICE)
_r = test_rows[:256]
_state = {"sequence": RAW_MAT[_r[:, None] + OFFSETS_NP], "position": 0, "profile": PROFILES["balanced"]}
_a, _b = policy.predict(_state), reloaded.predict(_state)
assert np.allclose(_a["action_probs"], _b["action_probs"], atol=1e-5) and (_a["action"] == _b["action"]).all(), "reload mismatch!"
_held = {**_state, "position": 1, "unrealized_pnl": 0.05, "holding_bars": 40}
assert np.allclose(policy.predict(_held)["action_probs"], reloaded.predict(_held)["action_probs"], atol=1e-5)
print("Round-trip check passed: reloaded policy reproduces in-memory predictions (flat and held states).")

ARTIFACTS = ["trend_following_policy.pt", "trend_following_config.json", "feature_schema.json", "preprocessing.pkl",
             "evaluation_metrics.json", "equity_curve.csv", "sp03_features.py", "sp03_model.py"]
for a in ARTIFACTS:
    assert (ART_DIR / a).exists(), f"missing artifact {a}"
    print(f"saved {(ART_DIR / a).resolve()}  ({(ART_DIR / a).stat().st_size / 1024:.1f} KiB)")

shutil.make_archive(str(ART_DIR), "zip", ART_DIR)
print(f"zip bundle: {Path(str(ART_DIR) + '.zip').resolve()}")
try:
    from google.colab import files  # noqa: F401  (only exists in Colab)
    files.download(str(ART_DIR) + ".zip")
except Exception:  # noqa: BLE001
    print("(not running in Colab or download blocked: copy the zip from the path above)")

## 12. Inference demonstration
Loads the saved policy **from disk** and predicts. It does **not** download data and does **not** train. The caller supplies a feature window, the position, the profile and, for held positions, unrealized P&L and holding bars.

In [ ]:
policy_loaded = TrendFollowingPolicy.load(ART_DIR, DEVICE)     # or TrendFollowingPolicy.load("path/to/trend_following_policy.pt", "cpu")
assert "yfinance" not in Path("sp03_model.py").read_text() and "yfinance" not in Path("sp03_features.py").read_text(), \
    "inference modules must not depend on a data downloader"

sym0 = list(raw_data)[0]
state = policy_loaded.state_from_ohlcv(raw_data[sym0], bench_df, position=0, profile={"risk_tolerance": 0.5, "patience": 0.5, "aggressiveness": 0.5})
out = policy_loaded.predict(state)
_reg_now = classify_regime({c: [state["sequence"][-1, i]] for i, c in enumerate(FEATURE_COLUMNS)}, CONFIG["regime"])[0]
print(f"{sym0} @ {raw_data[sym0].index[-1].date()} | regime = {REGIME_NAMES[_reg_now] if _reg_now >= 0 else 'unknown'} | flat | "
      f"action = {out['action_name']} (id {out['action']}) | confidence = {out['confidence']:.3f}")
print("action probabilities:", {n: round(float(p), 3) for n, p in zip(ACTION_NAMES, out["action_probs"])},
      "| valid:", {n: bool(v) for n, v in zip(ACTION_NAMES, out["valid_actions"])})

held = policy_loaded.predict({**state, "position": 1, "unrealized_pnl": 0.08, "holding_bars": 45})
print(f"same bar, LONG for 45 bars at +8.0% -> action = {held['action_name']} | confidence = {held['confidence']:.3f} | "
      f"valid: { {n: bool(v) for n, v in zip(ACTION_NAMES, held['valid_actions'])} }  (SELL is not valid while long; EXIT is the only way out)")

r = test_rows[:5]
batch = policy_loaded.predict({"sequence": RAW_MAT[r[:, None] + OFFSETS_NP], "position": [0, 0, 1, 1, -1],
                               "profile": PROFILES["conservative"], "unrealized_pnl": [0, 0, 0.04, -0.03, 0.02], "holding_bars": [0, 0, 30, 8, 12]})
print("batch actions:", batch["action_name"], "| confidences:", [round(float(c), 3) for c in batch["confidence"]])

## 13. Meta-RL integration checklist
Every SP-0x policy shares the action mapping, the execution convention, the state dictionary and the **base 30 sequence + 6 static columns**; each policy then appends its own extras, which **differ** (SP-02: mean-reversion extras, SP-03: trend extras). The Meta-RL environment should therefore build one feature table with the union of columns and let each policy select its columns **by name** from its `feature_schema.json` (not by position beyond the shared base).

In [ ]:
SP01_FEATURE_COLUMNS = ['ret_1', 'logret_1', 'ret_5', 'ret_10', 'ret_20', 'ret_60', 'rsi_14', 'macd_pct', 'macd_signal_pct', 'macd_hist_pct', 'close_ema20', 'close_ema50', 'ema20_ema50', 'close_sma200', 'sma50_sma200', 'adx_14', 'di_diff', 'atr_pct', 'rv_20', 'rv_60', 'vol_chg_1', 'vol_ratio_20', 'vol_z_60', 'hl_range', 'gap_open', 'close_loc', 'rel_strength_20', 'rel_strength_60', 'bench_ret_20', 'bench_rv_20']
SP02_STATIC = ['pos_flat', 'pos_long', 'pos_short', 'risk_tolerance', 'patience', 'aggressiveness', 'unrealized_pnl_scaled', 'holding_bars_scaled']
SP01_STATIC = ["pos_flat", "pos_long", "pos_short", "risk_tolerance", "patience", "aggressiveness"]
checks = {
    "action mapping is HOLD/BUY/SELL/EXIT = 0/1/2/3": ACTION_NAMES == ["HOLD", "BUY", "SELL", "EXIT"],
    "first 30 sequence columns identical to SP-01 and SP-02 (order and names)": FEATURE_COLUMNS[:len(SP01_FEATURE_COLUMNS)] == SP01_FEATURE_COLUMNS,
    "static input identical to SP-02 (8 dims) and prefix-identical to SP-01 (6 dims)": STATIC_FEATURE_NAMES == SP02_STATIC and STATIC_FEATURE_NAMES[:6] == SP01_STATIC,
    "loaded policy reproduces in-memory predictions": bool(np.allclose(_a["action_probs"], _b["action_probs"], atol=1e-5)),
    "inference modules contain no data downloader": "yfinance" not in Path("sp03_model.py").read_text() + Path("sp03_features.py").read_text(),
    "SELL is invalid while long/short; EXIT invalid while flat": bool(valid_action_mask([1], True)[0].tolist() == [True, False, False, True]
                                                                       and valid_action_mask([0], True)[0].tolist() == [True, True, True, False]),
}
assert all(checks.values()), f"compatibility check failed: {[k for k, v in checks.items() if not v]}"

print("=" * 100)
print("ARTIFACT PATHS")
for a in ARTIFACTS:
    print(f"  {(ART_DIR / a).resolve()}")
print(f"  {Path(str(ART_DIR) + '.zip').resolve()}   (zip of everything above)")
print("=" * 100)
print("META-RL INTEGRATION CHECKLIST: SP-03 Trend-Following Policy   ([x] = verified by an assertion in this notebook)")
for k in checks:
    print(f"  [x] {k}")
print( "  [ ] Build the union feature table and select columns by NAME per policy:")
print( "        cols = json.load(open('feature_schema.json'))['sequence_input']['features']; X = union_df[[c['name'] for c in cols]].to_numpy()")
print(f"        SP-03 sequence = (batch, {L}, {N_FEATURES}); static = {N_STATIC} dims; extras: {EXTRA_FEATURE_COLUMNS}")
print( "  [ ] Track per position in the environment: side, entry fill price (-> unrealized_pnl as a fraction), bars held (entry bar = 1).")
print( "  [ ] Call policy.predict({'sequence', 'position', 'profile', 'unrealized_pnl', 'holding_bars'}) -> action_probs(4), action, action_name, confidence, valid_actions, entropy.")
print( "  [ ] Respect execution timing: decide at the close of bar t, fill at the open of bar t+1 (the backtest here does exactly this).")
print( "  [ ] Treat EXIT as the only way to close a position; SELL only opens a short (and only if allow_short); invalid actions already have probability 0.")
print(f"  [ ] allow_short = {CONFIG['allow_short']} in this artifact; make the selector's execution layer consistent with it.")
print( "  [ ] Do NOT compare confidences across policies: each is calibrated against its own label rule, not against profit.")
print(f"  [ ] Labels: {LABEL_SOURCE}" + ("  -> strategy imitation, not human demonstrations" if LABEL_SOURCE.startswith("strategy_derived_proxy") else ""))
print( "  [ ] Optional: sp03_features.classify_regime(features) gives a past-only regime code the selector may use as context (it is NOT part of the policy input).")
print( "  [ ] Re-run walk-forward evaluation (e.g. retrain on rolling windows) before trusting any single train/validation/test split.")
print("-" * 100)
print("OUT-OF-SAMPLE (TEST) SNAPSHOT: one regime mix, one symbol list, costs included; NOT proof of profitability")
test_m = BT_TABLE["test"]
snap_rows = [MAIN] + ([LONG_ONLY] if LONG_ONLY in test_m.index else []) + [RULE, GC, f"{CONFIG['benchmark']} buy & hold", "Equal-weight universe buy & hold"]
print(test_m.loc[snap_rows, ["total_return", "sharpe", "max_drawdown", "annualized_turnover", "trade_entries"]].round(3).to_string())
print("-" * 100)
print("Research and paper-trading system only. Not a recommendation to trade real money. No profitability is guaranteed or implied.")
print("=" * 100)